# 02 · Ein Klick braucht keinen Aufsatz (Colab)
## Ein System-1-Entscheidungsmodell für Crawler trainieren

Ein LLM-Agent, der jede Seite liest und frei formuliert, was er als Nächstes tut, ist für Crawling über viele Sites
zu teuer und zu langsam. Die meisten Schritte sind aber keine Denkaufgabe, sondern eine **Auswahl**: Welcher der Links
auf dieser Seite bringt mich zum Ziel? Genau dafür trainieren wir hier ein kleines Entscheidungsmodell.

**Eingabe:** Auftrag, URL, bisherige Aktionen und der Accessibility-Baum der Seite.
**Ausgabe:** eine der ausführbaren Aktionen dieser Seite (Klick, Eingabe, Scrollen, Zurück, Stopp), bewertet in einem
einzigen Forward Pass über Buchstaben-Logits. Keine Textgenerierung.

| Baustein | Wahl |
|---|---|
| Basismodell | `Mapika/decider-2b` @ v10 (Commit `d61c1c1`, Qwen3.5-2B-Backbone, Option-Scoring über Label-Logits) |
| Daten | `stanfordnlp/nnetnav-live` @ `7f69dca`: echte Browser-Demonstrationen auf Live-Websites, BrowserGym-A11y-Bäume |
| Datenqualität | Auftragsfilter (Claude Haiku 4.5 über distilabel), Ergebnisse aus [`johannhartmann/nnetnav-live-objective-filter`](https://huggingface.co/datasets/johannhartmann/nnetnav-live-objective-filter) |
| Training | LoRA auf allen linearen Schichten, Cross-Entropy über die angebotenen Aktionen, **Colab-Größe** je nach GPU |
| Vergleich | der veröffentlichte Adapter [`johannhartmann/decider-2b-a11y-crawler-lora`](https://huggingface.co/johannhartmann/decider-2b-a11y-crawler-lora) (v2, alle 26.969 + 2.613 Entscheidungen, 29 h auf einer RTX A6000) und das Basismodell |
| Evaluation | dieselben Held-out-**Sites** (Amazon, Allrecipes, Cambridge Dictionary) und dieselbe Auswahl wie im vollen Lauf |
| Live | 16 Crawl-Aufgaben auf öffentlichen Scraping-Übungsseiten, Erfolg und extrahierte Daten |
| Korrekturschleife | Referenzabläufe korrigieren Fehlentscheidungen, Adapter wird damit weitertrainiert (optional) |

**Colab-Start:** nur dieses Notebook hochladen, *Laufzeit → Laufzeittyp ändern → GPU* (T4 reicht, L4/A100 sind
schneller und trainieren mehr), dann *Alle ausführen*. Alles Weitere kommt von PyPI, vom Hub und von GitHub
(BrowserGym-Skripte), jeweils auf festen Versionen bzw. Commits.

* **Profile** (`PROFILE = 'auto'` wählt nach GPU-Speicher): `t4` trainiert 2.000 Aktions- + 200 Texteingabe-Entscheidungen
  mit Seiten bis 8.192 Tokens, `l4` 4.000 + 400 bis 12.288 Tokens, `a100` 8.000 + 800 bis 16.384 Tokens. `full` ist das Rezept
  des veröffentlichten Adapters (nur manuell; in Colab praktisch nicht machbar). Die geschätzte und die gemessene
  Restzeit werden beim Training ausgegeben.
* **GPU:** L4/A100 rechnen in BF16. Eine T4 kann kein natives BF16; `PRECISION = 'auto'` schaltet dort auf FP16 (Gewichte FP16,
  LoRA-Parameter FP32, Loss-Scaling). Abschnitt 9 vergleicht die Entscheidungen des veröffentlichten Adapters in dieser
  Laufzeit Entscheidung für Entscheidung mit dem BF16-Referenzlauf.
* **Kosten:** Bezahlte API-Schritte sind aus (`ALLOW_PAID_API = False`). Der Auftragsfilter kommt fertig vom Hub.
* **Secrets** (Schlüsselsymbol links, optional): `HF_TOKEN` mit Schreibrecht für den Upload des eigenen Adapters,
  `ANTHROPIC_API_KEY` nur für die optionale Neuerzeugung des Filters.
* **Google Drive** (`USE_DRIVE`, Standard an): Daten-Indizes, Checkpoints, Adapter und Auswertungen liegen in
  `MyDrive/a11y-training`. Nach einem Abbruch einfach erneut *Alle ausführen*: Das Laufverzeichnis ergibt sich aus Profil
  und Rezept, das Training setzt am letzten Checkpoint fort, fertige Teile werden geladen statt neu gerechnet.

## 1 · Umgebung
`decider-ai` pinnt `numpy<2`, benutzt für Prompt und Readout aber kein NumPy. Wir installieren das Paket deshalb ohne
Abhängigkeiten und die tatsächlich benötigten Pakete explizit – Colabs NumPy 2 bleibt unangetastet, ein Neustart der
Laufzeit ist nicht nötig. Chromium für Playwright kommt mit seinen Systembibliotheken (`--with-deps`, apt).

In [ ]:
# @title Pakete installieren (gepinnt), Chromium für Playwright
import os, sys, shutil, subprocess, importlib, importlib.metadata as md

if sys.version_info < (3, 11):
    raise RuntimeError('Python 3.11 oder neuer erforderlich.')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('DISTILABEL_LOG_LEVEL', 'WARNING')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # weniger Fragmentierung bei wechselnden Längen
# causal-conv1d: CUDA-Kernel für die kurze Faltung der Gated-DeltaNet-Schichten (+4-6 % Durchsatz). Ohne vorgebautes
# Wheel wird er kompiliert (auf Colab lange); ohne ihn rechnet transformers dieselbe Faltung in PyTorch.
INSTALL_CAUSAL_CONV1D = False  # @param {type:'boolean'}

PINNED = {'decider-ai': '1.5.0', 'transformers': '5.17.0', 'peft': '0.21.0', 'flash-linear-attention': '0.5.2',
          'playwright': '1.57.0', 'distilabel': '1.5.3'}
NUMPY_BEFORE = md.version('numpy')
loaded_before = {m: sys.modules[m].__version__ for m in ('transformers', 'peft', 'huggingface_hub')
                 if m in sys.modules and hasattr(sys.modules[m], '__version__')}


def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])


pip('--no-deps', f"decider-ai=={PINNED['decider-ai']}")
pip(*[f'{k}=={v}' for k, v in PINNED.items() if k != 'decider-ai'],
    'accelerate>=1.3,<2', 'huggingface_hub>=1,<2', 'jinja2', 'anthropic>=1.8', 'requests>=2.32', 'matplotlib>=3.8')
if INSTALL_CAUSAL_CONV1D:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'causal-conv1d'],
                       capture_output=True, text=True)
    print('causal-conv1d', 'installiert' if r.returncode == 0 else 'nicht installiert – PyTorch-Faltung wird benutzt')
importlib.invalidate_caches()

versions = {k: md.version(k) for k in PINNED}
assert all(versions[k] == v for k, v in PINNED.items()), versions
assert md.version('numpy') == NUMPY_BEFORE, f"numpy wurde von {NUMPY_BEFORE} auf {md.version('numpy')} geändert"
stale = {m: v for m, v in loaded_before.items() if v != md.version(m.replace('_', '-'))}
if stale:
    raise RuntimeError(f'Schon vor der Installation geladen: {stale}. Laufzeit → Sitzung neu starten, dann diese Zelle erneut.')

# Chromium + Systembibliotheken. Colab ist Ubuntu: --with-deps installiert die Bibliotheken per apt.
if shutil.which('apt-get'):
    subprocess.check_call([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'])
else:
    print('Kein apt-get (kein Debian/Ubuntu): nur Chromium wird geladen, die Systembibliotheken müssen vorhanden sein.')
    subprocess.check_call([sys.executable, '-m', 'playwright', 'install', 'chromium'])
print('numpy', md.version('numpy'), '|', versions)
print('Die pip-Meldung "decider-ai requires numpy<2" ist erwartet: der Pin wird bewusst übergangen (siehe oben).')

In [ ]:
# @title Colab: Drive, Secrets, Pfade, Hilfsmodul vom Hub
import hashlib, json
from pathlib import Path
from huggingface_hub import hf_hub_download

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True         # @param {type:'boolean'}   Colab: Laufverzeichnis auf Google Drive (Fortsetzen nach Abbruch)
OUTPUT_ROOT = ''         # @param {type:'string'}    leer = MyDrive/a11y-training bzw. ./a11y-training
NNETNAV_ON_DRIVE = False # @param {type:'boolean'}   train.jsonl (3,2 GB) zusätzlich auf Drive cachen statt jedes Mal zu laden


def mount_drive():
    # Drive einbinden; abgelehnt oder außerhalb von Colab -> None (Ausgaben lokal).
    if not IN_COLAB:
        return None
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        return Path('/content/drive/MyDrive')
    except Exception as exc:
        print('Drive nicht eingebunden (', type(exc).__name__, '): Ausgaben gehen mit der Sitzung verloren.')
        return None


def colab_secret(name):
    # Umgebung hat Vorrang, dann Colab-Secret. Der Wert wird nie ausgegeben.
    if os.environ.get(name):
        return os.environ[name]
    if not IN_COLAB:
        return None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception:  # Secret fehlt oder kein Notebook-Zugriff
        return None


if not OUTPUT_ROOT:
    drive_root = mount_drive() if USE_DRIVE else None
    OUTPUT_ROOT = str(drive_root / 'a11y-training') if drive_root else ('/content/a11y-training' if IN_COLAB else './a11y-training')
ROOT = Path(OUTPUT_ROOT).expanduser().resolve()
ROOT.mkdir(parents=True, exist_ok=True)
for name in ['HF_TOKEN', 'ANTHROPIC_API_KEY']:
    value = colab_secret(name)
    if value:
        os.environ[name] = value
HF_TOKEN = os.environ.get('HF_TOKEN') or None
HAVE_API_KEY = bool(os.environ.get('ANTHROPIC_API_KEY'))

# Hilfsmodul des Auftragsfilters aus dem öffentlichen Modell-Repo (fester Commit von v2, Prüfsumme = Stand im
# Projekt-Repo). Es enthält die Sicherheitskorrektur für distilabel: `import distilabel` installiert Rich-Tracebacks mit
# show_locals=True, die bei einem Fehler lokale Variablen (also auch Tokens) ausgeben würden; die vorherigen Handler
# werden sofort wiederhergestellt.
MODEL_REPO = 'johannhartmann/decider-2b-a11y-crawler-lora'
PUBLISHED_REVISION = '39421be6b01a6380f2811a32fbc57f92963af930'   # main = v2
FILTER_MODULE_SHA256 = '0b06b5f11c6abaf91d5f0f3bdbabb482905387a19b6828b8377f49dea5d39287'
module_text = Path(hf_hub_download(MODEL_REPO, 'training/02_data/objective_filter.py', revision=PUBLISHED_REVISION)).read_text()
assert hashlib.sha256(module_text.encode()).hexdigest() == FILTER_MODULE_SHA256, 'objective_filter.py auf dem Hub weicht ab'
assert 'def _import_distilabel_without_rich_locals' in module_text and 'sys.excepthook = hook' in module_text
module_dir = ROOT / 'code'
module_dir.mkdir(exist_ok=True)
(module_dir / 'objective_filter.py').write_text(module_text)
sys.path.insert(0, str(module_dir))
print('Colab:', IN_COLAB, '| Ausgaben:', ROOT, '| HF-Token:', bool(HF_TOKEN), '| Anthropic-Key:', HAVE_API_KEY)

## 2 · Profil, Präzision und Laufverzeichnis
Die **Held-out-Auswahl ist immer dieselbe** wie im vollen Lauf (600 Aktions- und 400 Texteingabe-Entscheidungen, Seiten bis
24.576 Tokens, Fingerabdruck wird gegen `decision_contract.json` des veröffentlichten Adapters geprüft). Kleinere Profile werten
die ersten `n_eval` davon aus – bei jedem Lauf dieselben. Ebenso bleibt die Aufteilung in Trainings- und Validierungsaufgaben
gleich; das Profil bestimmt nur, wie viele Trainingsentscheidungen (und bis zu welcher Seitenlänge) zufällig gezogen werden.

| Profil | Training (Aktion + Text) | Seiten bis | Held-out ausgewertet | Validierung | Auto-Wahl |
|---|---|---|---|---|---|
| `t4` | 2.000 + 200, max. 2 je Aufgabe | 8.192 Tokens | 300 von 600 (+400 Texte) | 60 | Standard |
| `l4` | 4.000 + 400, max. 3 je Aufgabe | 12.288 | 600 (+400) | 100 | ≥ 20 GiB GPU |
| `a100` | 8.000 + 800 | 16.384 | 600 (+400) | 150 | ≥ 38 GiB GPU |
| `full` | alle (26.969 + 2.613) | 24.576 | 600 (+400) | 300 | nur manuell |

In [ ]:
# @title Profil und Konfiguration
PROFILE = 'auto'      # @param ['auto', 't4', 'l4', 'a100', 'full']
PRECISION = 'auto'    # @param ['auto', 'bf16', 'fp16']   auto: BF16 ab Ampere (L4/A100), sonst FP16 (T4)
RUN_CORRECTIONS = True    # @param {type:'boolean'}   Abschnitt 11: Korrekturen sammeln und weitertrainieren
LIVE_BASE = 'auto'        # @param ['auto', 'ja', 'nein']   Basismodell auch live crawlen (auto: nicht auf t4)
ALLOW_PAID_API = False    # @param {type:'boolean'}   Anthropic-API erlauben (nur für die Neuerzeugung des Filters)
PUSH_ADAPTER = True       # @param {type:'boolean'}   eigenen Adapter hochladen (braucht HF_TOKEN mit Schreibrecht)
HUB_ADAPTER_REPO = ''     # @param {type:'string'}    leer = <dein-user>/decider-2b-a11y-crawler-lora-colab
HUB_ADAPTER_PRIVATE = True  # @param {type:'boolean'}

import random, shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Keine GPU. Laufzeit → Laufzeittyp ändern → GPU (T4, L4 oder A100).')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
RAM_GIB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
DISK_FREE_GIB = shutil.disk_usage(ROOT).free / 2**30

PROFILES = {
    't4':   dict(n_train=2000, n_value_train=200, per_task=2, max_tokens=8192, n_eval=300, n_val=60, n_value_val=40,
                 eval_every=100, save_every=25, n_regression=100, live_base=False),
    'l4':   dict(n_train=4000, n_value_train=400, per_task=3, max_tokens=12288, n_eval=600, n_val=100, n_value_val=60,
                 eval_every=200, save_every=25, n_regression=300, live_base=True),
    'a100': dict(n_train=8000, n_value_train=800, per_task=None, max_tokens=16384, n_eval=600, n_val=150, n_value_val=100,
                 eval_every=300, save_every=50, n_regression=300, live_base=True),
    'full': dict(n_train=None, n_value_train=None, per_task=None, max_tokens=24576, n_eval=600, n_val=300, n_value_val=200,
                 eval_every=500, save_every=50, n_regression=300, live_base=True),
}
COMMON = dict(epochs=1, lr=1e-4, rank=16, accumulation=8, objective_filter=True, n_val_tasks=160, max_steps=12)
if PROFILE == 'auto':
    PROFILE = 'a100' if GPU_GIB >= 38 else 'l4' if GPU_GIB >= 20 else 't4'
CFG = {**COMMON, **PROFILES[PROFILE]}
if LIVE_BASE != 'auto':
    CFG['live_base'] = LIVE_BASE == 'ja'
if DISK_FREE_GIB < 15:
    raise RuntimeError(f'Nur {DISK_FREE_GIB:.0f} GiB frei unter {ROOT}; gebraucht werden etwa 15 GiB.')

# torch.cuda.is_bf16_supported() meldet auf einer T4 (sm75) wegen Emulation True, deshalb Compute Capability prüfen.
NATIVE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
PRECISION = PRECISION if PRECISION != 'auto' else ('bf16' if NATIVE_BF16 else 'fp16')
if PRECISION == 'bf16' and not NATIVE_BF16:
    raise RuntimeError(f'{GPU_NAME} hat kein natives BF16: PRECISION auf fp16 oder auto setzen.')
DTYPE = {'bf16': torch.bfloat16, 'fp16': torch.float16}[PRECISION]

SEED = 41
MODEL_ID, MODEL_SHA = 'Mapika/decider-2b', 'd61c1c16089572df5d180329b9fea4997a90090c'            # v10
DATASET_ID, DATASET_SHA = 'stanfordnlp/nnetnav-live', '7f69dca22bb2faaf71333b29c2ce0fbf8af39de8'
FILTER_REPO, FILTER_REVISION = 'johannhartmann/nnetnav-live-objective-filter', '80eefb05e0d291b6b1e3f040a6cabb2401b2c6b7'
HELDOUT_SITES = {'amazon.com', 'allrecipes.com', 'cambridge.org'}
EVAL_MAX_TOKENS, N_EVAL_ALL, N_VALUE_EVAL = 24576, 600, 400     # Held-out-Auswahl wie im vollen Lauf
RECIPE_VERSION = 'nb2-colab-v1'

recipe_key = hashlib.sha256(json.dumps([CFG, SEED, MODEL_SHA, sorted(HELDOUT_SITES), RECIPE_VERSION, FILTER_REVISION,
                                        PRECISION], sort_keys=True).encode()).hexdigest()[:10]
RUN_DIR = ROOT / 'runs' / f'{PROFILE}-{PRECISION}-{recipe_key}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
NN_DIR = ROOT / 'nnetnav' / DATASET_SHA[:10]     # profilunabhängig: Metadaten und Held-out-Auswahl
NN_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR, CONTINUED_DIR = RUN_DIR / 'adapter', RUN_DIR / 'adapter-continued'
if ALLOW_PAID_API:
    assert HAVE_API_KEY, 'ALLOW_PAID_API braucht das Colab-Secret ANTHROPIC_API_KEY.'
print(f'GPU: {GPU_NAME} ({GPU_GIB:.0f} GiB) | RAM {RAM_GIB:.0f} GiB | Platte frei {DISK_FREE_GIB:.0f} GiB')
print('Profil:', PROFILE, '| Präzision:', PRECISION, '|', CFG)
print('Laufverzeichnis:', RUN_DIR, '(vorhandene Zwischenstände werden weiterverwendet)')

In [ ]:
# @title Imports, feste Revisionen
import ast, asyncio, base64, collections, gc, math, re, struct, time
from contextlib import contextmanager, nullcontext
from types import SimpleNamespace
from urllib.parse import urlparse

import pandas as pd
import requests
import torch.nn.functional as F
from IPython.display import display
from tqdm.auto import tqdm
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainerCallback, TrainingArguments, set_seed
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from decider.prompt import build as build_decision_prompt, letter_ids
from decider.model import collate as collate_decisions
from playwright.async_api import async_playwright

if PRECISION == 'fp16':
    print('FP16-Modus (z. B. T4): Gewichte in FP16, LoRA-Parameter in FP32, Training mit Loss-Scaling.')
DEVICE = torch.device('cuda:0')
set_seed(SEED)
torch.backends.cuda.enable_cudnn_sdp(False)   # wie decider.engine: cuDNN-SDPA ist bei Masken unzuverlässig
# Ohne Flash-Kernel (T4 = sm75) rechnet PyTorch-SDPA Grouped-Query-Attention (8 Query-, 2 KV-Köpfe) nur im quadratischen
# Referenzkernel: bei 24k Tokens rund 9 GiB Attention-Gewichte je Schicht. Mit expandierten KV-Köpfen nimmt SDPA den
# speichersparenden Kernel (linearer Speicher, mathematisch identisch).
FLASH_SDPA = torch.cuda.get_device_capability(0) >= (8, 0) and torch.backends.cuda.flash_sdp_enabled()
if not FLASH_SDPA:
    import transformers.integrations.sdpa_attention as _sdpa
    _sdpa.use_gqa_in_sdpa = lambda attention_mask, key, value: False
    print('Kein Flash-Kernel: KV-Köpfe werden expandiert, SDPA nutzt den speichersparenden Kernel.')

hub = HfApi()
assert hub.model_info(MODEL_ID, revision=MODEL_SHA).sha == MODEL_SHA
PUBLISHED_CONTRACT = json.loads(Path(hf_hub_download(MODEL_REPO, 'decision_contract.json', revision=PUBLISHED_REVISION)).read_text())
assert PUBLISHED_CONTRACT['base_revision'] == MODEL_SHA and PUBLISHED_CONTRACT['version'] == 'v2'
assert PUBLISHED_CONTRACT['training_data']['revision'] == DATASET_SHA
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_SHA)
if TOKENIZER.pad_token_id is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
TIMINGS = {}


def pkg_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


versions = {p: pkg_version(p) for p in ['torch', 'transformers', 'peft', 'decider-ai', 'playwright',
                                        'flash-linear-attention', 'causal-conv1d', 'distilabel', 'numpy']}


def wait_for_gpu(min_free_gib, timeout_s=6 * 3600):
    # Auf einer geteilten GPU warten, bis genug Speicher frei ist (eigener Cache zählt als frei).
    gc.collect(); torch.cuda.empty_cache()
    start, told = time.time(), False
    while torch.cuda.mem_get_info()[0] / 2**30 < min_free_gib:
        if not told:
            print(f'Warte auf {min_free_gib} GiB freien GPU-Speicher ...', flush=True); told = True
        if time.time() - start > timeout_s:
            raise RuntimeError(f'Weniger als {min_free_gib} GiB GPU-Speicher frei.')
        time.sleep(30)


print('Modell:', MODEL_ID, MODEL_SHA[:10], '| Daten:', DATASET_ID, DATASET_SHA[:10], '| Vergleich:', MODEL_REPO,
      PUBLISHED_REVISION[:10], f"({PUBLISHED_CONTRACT['version']})")
print(versions)
print('Kernels: Gated DeltaNet über flash-linear-attention', '✓' if versions['flash-linear-attention'] else '✗',
      '| kurze Faltung über causal-conv1d', '✓' if versions['causal-conv1d'] else '✗ (PyTorch-Faltung)')

## 3 · Beobachtung: derselbe A11y-Baum wie in den Trainingsdaten
NNetNav wurde mit BrowserGym/AgentLab aufgezeichnet: Jedes DOM-Element bekommt eine `bid`, Sichtbarkeit
kommt aus einem IntersectionObserver, `clickable` aus dem DOMSnapshot von Chromium, und
`flatten_axtree_to_str` schreibt den Baum mit Tabs, `repr`-Namen und den Flags `clickable, visible`.
Ein Modell, das auf diesem Format trainiert wird, muss live exakt dieses Format sehen.

Deshalb laden wir BrowserGyms Markierungsskripte unverändert vom Tag `v0.13.3` (SHA-256-geprüft) und
portieren die Python-Extraktion 1:1 auf Playwright-async. Einzige Abweichung: aktuelle Chromium-Versionen
liefern zusätzlich `InlineTextBox`-Knoten, die im Datensatz nie vorkommen; sie werden ignoriert.

In [ ]:
# @title BrowserGym-Extraktion (Port von v0.13.3, Apache-2.0)
BGYM_TAG = 'v0.13.3'
BGYM_JS_FILES = {
    'mark': ('browsergym/core/src/browsergym/core/javascript/frame_mark_elements.js',
             '6ecfbba89d756c37874bdf692e4bf24126459ab79f9120c1f053305b7ba2978a'),
    'unmark': ('browsergym/core/src/browsergym/core/javascript/frame_unmark_elements.js',
               'ee1324efd8df5c12693fba0c4814ecb829ef92860d972e5377bdd5c905b28526'),
}
BID_ATTR, VIS_ATTR, SOM_ATTR = 'bid', 'browsergym_visibility_ratio', 'browsergym_set_of_marks'


def load_bgym_js(cache_dir):
    out = {}
    for key, (path, sha) in BGYM_JS_FILES.items():
        f = Path(cache_dir) / f'bgym-{BGYM_TAG}-{Path(path).name}'
        if not f.exists():
            r = requests.get(f'https://raw.githubusercontent.com/ServiceNow/BrowserGym/{BGYM_TAG}/{path}', timeout=60)
            r.raise_for_status()
            f.write_bytes(r.content)
        data = f.read_bytes()
        if hashlib.sha256(data).hexdigest() != sha:
            raise RuntimeError(f'{path}: unerwarteter Inhalt')
        out[key] = data.decode()
    return out


async def mark_frames(page, js):
    async def rec(frame, frame_bid):
        await frame.evaluate(js['mark'], [frame_bid, BID_ATTR, 'standard_html'])
        for child in frame.child_frames:
            if child.is_detached():
                continue
            elem = await child.frame_element()
            if await elem.content_frame() != child:
                continue
            sandbox = await elem.get_attribute('sandbox')
            if sandbox is not None and 'allow-scripts' not in sandbox.split():
                continue
            child_bid = await elem.get_attribute(BID_ATTR)
            if child_bid is not None:
                await rec(child, child_bid)
    await rec(page.main_frame, '')


async def unmark_frames(page, js):
    for frame in page.frames:
        try:
            if frame != page.main_frame:
                elem = await frame.frame_element()
                if await elem.content_frame() != frame or await elem.get_attribute(BID_ATTR) is None:
                    continue
                sandbox = await elem.get_attribute('sandbox')
                if sandbox is not None and 'allow-scripts' not in sandbox.split():
                    continue
            await frame.evaluate(js['unmark'])
        except Exception as exc:
            if 'detached' not in str(exc):
                raise


_ARIA_DATA = re.compile(r'^browsergym_id_([a-zA-Z0-9]+)\s?(.*)')


def _aria_data(s):
    m = _ARIA_DATA.fullmatch(s)
    return ([m.group(1)], m.group(2)) if m else ([], s)


def _pop_bids(snapshot, attr):
    if attr not in snapshot['strings']:
        return
    target, done = snapshot['strings'].index(attr), set()
    for document in snapshot['documents']:
        for attrs in document['nodes']['attributes']:
            for i in range(0, len(attrs), 2):
                if attrs[i] == target:
                    vid = attrs[i + 1]
                    value = snapshot['strings'][vid]
                    if vid not in done:
                        _, value = _aria_data(value)
                        snapshot['strings'][vid] = value
                        done.add(vid)
                    if value == '':
                        del attrs[i:i + 2]
                    break


async def extract_extra_properties(page):
    cdp = await page.context.new_cdp_session(page)
    try:
        snap = await cdp.send('DOMSnapshot.captureSnapshot',
                              {'computedStyles': [], 'includeDOMRects': True, 'includePaintOrder': True})
    finally:
        await cdp.detach()
    _pop_bids(snap, 'aria-roledescription')
    _pop_bids(snap, 'aria-description')
    strings = snap['strings']
    sid = {n: (strings.index(n) if n in strings else -1) for n in (BID_ATTR, VIS_ATTR, SOM_ATTR)}
    docs, todo = {0: {'parent': None}}, [0]
    while todo:
        doc = todo.pop(-1)
        children = snap['documents'][doc]['nodes']['contentDocumentIndex']
        for node, child in zip(children['index'], children['value']):
            docs[child] = {'parent': {'doc': doc, 'node': node}}
            todo.append(child)
        parent, px, py = docs[doc]['parent'], 0, 0
        if parent:
            layout = snap['documents'][parent['doc']]['layout']
            if parent['node'] in layout['nodeIndex']:
                b = layout['bounds'][layout['nodeIndex'].index(parent['node'])]
                px, py = docs[parent['doc']]['abs_pos']['x'] + b[0], docs[parent['doc']]['abs_pos']['y'] + b[1]
        document = snap['documents'][doc]
        docs[doc]['abs_pos'] = {'x': px - document['scrollOffsetX'], 'y': py - document['scrollOffsetY']}
        nodes = [{'bid': None, 'visibility': None, 'bbox': None, 'clickable': False, 'set_of_marks': None}
                 for _ in document['nodes']['parentIndex']]
        for i in document['nodes']['isClickable']['index']:
            nodes[i]['clickable'] = True
        for i, attrs in enumerate(document['nodes']['attributes']):
            for j in range(0, len(attrs), 2):
                if attrs[j] == sid[BID_ATTR]:
                    nodes[i]['bid'] = strings[attrs[j + 1]]
                elif attrs[j] == sid[VIS_ATTR]:
                    nodes[i]['visibility'] = float(strings[attrs[j + 1]])
                elif attrs[j] == sid[SOM_ATTR]:
                    nodes[i]['set_of_marks'] = strings[attrs[j + 1]] == '1'
        for i, bounds, rect in zip(document['layout']['nodeIndex'], document['layout']['bounds'],
                                   document['layout']['clientRects']):
            if rect:
                nodes[i]['bbox'] = [bounds[0] + docs[doc]['abs_pos']['x'], bounds[1] + docs[doc]['abs_pos']['y'],
                                    bounds[2], bounds[3]]
        docs[doc]['nodes'] = nodes
    return {n['bid']: {k: n[k] for k in ('visibility', 'bbox', 'clickable', 'set_of_marks')}
            for d in docs.values() for n in d['nodes'] if n['bid']}


async def extract_merged_axtree(page):
    cdp = await page.context.new_cdp_session(page)
    try:
        frames, todo = [], [(await cdp.send('Page.getFrameTree', {}))['frameTree']]
        while todo:
            frame = todo.pop()
            todo.extend(frame.get('childFrames', []))
            frames.append(frame['frame']['id'])
        axtrees = {fid: await cdp.send('Accessibility.getFullAXTree', {'frameId': fid}) for fid in frames}
        for ax in axtrees.values():
            for node in ax['nodes']:
                items = []
                for i, prop in enumerate(node.get('properties', [])):
                    if prop['name'] == 'roledescription':
                        items, new = _aria_data(prop['value']['value'])
                        prop['value']['value'] = new
                        if new == '':
                            del node['properties'][i]
                        break
                if 'description' in node:
                    items2, new = _aria_data(node['description']['value'])
                    node['description']['value'] = new
                    if new == '':
                        del node['description']
                    items = items or items2
                if items:
                    node['browsergym_id'] = items[0]
        merged = {'nodes': []}
        for ax in axtrees.values():
            merged['nodes'].extend(ax['nodes'])
            for node in ax['nodes']:
                if node['role']['value'] == 'Iframe':
                    fid = (await cdp.send('DOM.describeNode', {'backendNodeId': node['backendDOMNodeId']})).get('node', {}).get('frameId')
                    if fid in axtrees:
                        node['childIds'].append(axtrees[fid]['nodes'][0]['nodeId'])
    finally:
        await cdp.detach()
    return merged


IGNORED_ROLES = {'LineBreak', 'InlineTextBox'}   # InlineTextBox: nur neuere Chromium-Versionen, nie im Datensatz
IGNORED_PROPERTIES = ('editable', 'readonly', 'level', 'settable', 'multiline', 'invalid', 'focusable')


def flatten_axtree_to_str(ax_tree, extra_properties):
    """browsergym.utils.obs.flatten_axtree_to_str mit AgentLabs NNetNav-Flags (with_visible, with_clickable)."""
    index = {n['nodeId']: i for i, n in enumerate(ax_tree['nodes'])}

    def dfs(i, depth, parent_filtered, parent_name):
        out, node = '', ax_tree['nodes'][i]
        skip, role, name = False, node['role']['value'], ''
        if role in IGNORED_ROLES or 'name' not in node:
            skip = True
        else:
            name = node['name']['value']
            value = node['value']['value'] if 'value' in node and 'value' in node['value'] else None
            bid, attrs = node.get('browsergym_id'), []
            for prop in node.get('properties', []):
                if 'value' not in prop or 'value' not in prop['value'] or prop['name'] in IGNORED_PROPERTIES:
                    continue
                if prop['name'] in ('required', 'focused', 'atomic'):
                    if prop['value']['value']:
                        attrs.append(prop['name'])
                else:
                    attrs.append(f"{prop['name']}={prop['value']['value']!r}")
            if role == 'generic' and not attrs:
                skip = True
            if role == 'StaticText':
                skip = skip or parent_filtered or name in parent_name
            elif bid is not None and bid in extra_properties:
                p = extra_properties[bid]
                attrs = (['clickable'] if p['clickable'] else []) + \
                        (['visible'] if (p['visibility'] or 0) >= 0.5 else []) + attrs
            if not skip:
                s = role if (role == 'generic' and not name) else f'{role} {name.strip()!r}'
                if bid is not None:
                    s = f'[{bid}] ' + s
                if value is not None:
                    s += f' value={value!r}'
                out = '\t' * depth + s + ''.join(', ' + a for a in attrs)
        for child in node['childIds']:
            if child in index and child != node['nodeId']:
                sub = dfs(index[child], depth if skip else depth + 1, False, name)
                if sub:
                    out += ('\n' if out else '') + sub
        return out

    return dfs(0, 0, False, '')


async def bgym_observe(page, js):
    await mark_frames(page, js)
    try:
        ax = await extract_merged_axtree(page)
        extra = await extract_extra_properties(page)
    finally:
        await unmark_frames(page, js)
    return flatten_axtree_to_str(ax, extra), ax, extra


BGYM_JS = load_bgym_js(ROOT)
print('BrowserGym-Skripte', BGYM_TAG, 'geladen und geprüft.')

## 4 · Aktionsraum
Aus jedem Baum entstehen dieselben Kandidaten – im Training, in der Evaluation und live:

* **click** auf jedes Element mit interaktiver Rolle oder Chromiums `clickable`-Flag,
* **type** in jedes Eingabefeld (`textbox`, `searchbox`, `combobox`), mit und ohne Enter,
* `scroll [down]`, `scroll [up]`, `go_back`, `stop`.

Deciders Label-Kopf hat 255 Plätze. Hat eine Seite mehr Kandidaten, werden nur sichtbare Elemente
angeboten – dieselbe Regel gilt im Training. `hover` (0,9 % der Demonstrationen), Tastenkürzel, Tabs und
`goto` sind nicht Teil des Aktionsraums.

**Texteingaben.** Ein Auswahlmodell kann keinen Text erfinden. Die Eingabe ist daher eine zweite,
kleine Entscheidung: Welcher Text aus dem Auftrag gehört in das Feld? Kandidaten sind zitierte Werte,
alle zusammenhängenden Spannen des Auftrags, deren Stichwort-Verdichtung (Füllwörter entfernt) und
früher eingegebene Werte. Wie oft der demonstrierte Text darunter ist, messen wir unten.

Die Historie hat exakt NNetNavs Format (`2: click [4189] where [4189] is Dismiss sign-in info.`).

In [ ]:
# @title Kandidaten, Texteingaben, Historie
NODE_RE = re.compile(r"^(\t*)\[([A-Za-z0-9]+)\] ([A-Za-z-]+) (?:'((?:[^'\\]|\\.)*)'|\"((?:[^\"\\]|\\.)*)\")(.*)$")
CLICK_ROLES = {'link', 'button', 'checkbox', 'radio', 'tab', 'menuitem', 'menuitemcheckbox', 'menuitemradio',
               'option', 'switch', 'combobox', 'textbox', 'searchbox', 'listbox', 'treeitem', 'gridcell',
               'DisclosureTriangle', 'slider', 'spinbutton'}
TYPE_ROLES = {'textbox', 'searchbox', 'combobox'}
MAX_OPTIONS = 255  # Breite von Deciders Label-Kopf


def parse_tree(tree):
    """Elementzeilen eines BrowserGym-Baums: bid -> dict(role, name, flags, disabled)."""
    nodes = {}
    for line in tree.split('\n'):
        m = NODE_RE.match(line)
        if not m or m[2] in nodes:
            continue
        raw, quote = (m[4], "'") if m[4] is not None else (m[5], '"')
        try:
            name = ast.literal_eval(quote + raw + quote)
        except (ValueError, SyntaxError):
            name = raw
        rest = m[6]
        flags = {p.strip() for p in rest.split(', ') if p.strip() and '=' not in p}
        nodes[m[2]] = dict(role=m[3], name=name, flags=flags, disabled='disabled=True' in rest)
    return nodes


def short(text, n=70):
    text = ' '.join(str(text).split())
    return text if len(text) <= n else text[:n - 1] + '…'


def action_key(a):
    return {'click': ('click', a.get('bid')), 'type': ('type', a.get('bid'), a.get('enter')),
            'scroll': ('scroll', a.get('direction'))}.get(a['kind'], (a['kind'],))


def candidates(tree):
    """Alle ausführbaren Aktionen eines Zustands. Wird für Training, Evaluation und Live-Browser identisch benutzt."""
    nodes = parse_tree(tree)
    def build(only_visible):
        acts = []
        for bid, n in nodes.items():
            if n['disabled'] or (only_visible and 'visible' not in n['flags']):
                continue
            if n['role'] in CLICK_ROLES or 'clickable' in n['flags']:
                acts.append(dict(kind='click', bid=bid, label=f"click [{bid}] {n['role']} {short(n['name'])!r}"))
            if n['role'] in TYPE_ROLES:
                for enter in (True, False):
                    acts.append(dict(kind='type', bid=bid, enter=enter,
                                     label=f"type into [{bid}] {n['role']} {short(n['name'])!r}"
                                           + (' and press Enter' if enter else ' without pressing Enter')))
        acts += [dict(kind='scroll', direction='down', label='scroll [down]'),
                 dict(kind='scroll', direction='up', label='scroll [up]'),
                 dict(kind='go_back', label='go_back'),
                 dict(kind='stop', label='stop: objective reached or not reachable')]
        return acts
    acts = build(False)
    if len(acts) > MAX_OPTIONS:          # sehr große Seiten: nur sichtbare Elemente anbieten (Training und Live gleich)
        acts = build(True)
    return acts, nodes


STOPWORDS = set('a an the of for to in on at by with and or from about into near its their his her my your our '
                'this that these those is are be as what which who how when where'.split())
FILLER = STOPWORDS | set('made using use related regarding including include includes find search look get learn '
                         'explore research check browse information info details detail some any all more most '
                         'specific specifically particular certain other different various'.split())


def norm_text(s):
    s = re.sub(r'\s+', ' ', s.lower().replace('’', "'")).strip()
    return s.strip(' "\'.,;:!?()[]')


def value_candidates(goal, history):
    """Kandidaten für Texteingaben: zitierte Werte, zusammenhängende Spannen und Stichwort-Verdichtungen des Auftrags
    sowie früher eingegebene Werte. Die richtige Aktion wird dafür nie gelesen."""
    words = goal.replace('“', '"').replace('”', '"').split()
    toks = [w.strip('"\'.,;:!?()[]') for w in words]
    out = list(re.findall(r'"([^"]{1,120})"', goal.replace('“', '"').replace('”', '"')))
    out += re.findall(r"(?<![\w])'([^']{2,120})'(?![\w])", goal)
    for i in range(len(toks)):
        for j in range(i + 1, min(len(toks), i + 10) + 1):
            span = [t for t in toks[i:j] if t]
            if not span or span[0].lower() in STOPWORDS or span[-1].lower() in STOPWORDS:
                continue
            if j - i <= 8:
                out.append(' '.join(span))
            dense = [t for t in span if t.lower() not in FILLER]
            if dense and len(dense) <= 6:
                out.append(' '.join(dense))
                last = dense[-1]
                if len(last) > 3 and last.endswith('s') and not last.endswith('ss'):
                    out.append(' '.join(dense[:-1] + [last[:-1]]))
    for line in history:
        m = re.match(r'(?:\d+: )?type \[[^\]]+\] \[(.*?)\s?\] where', line)
        if m and m[1].strip():
            out.append(m[1].strip())
    seen, values = set(), []
    for v in out:
        k = norm_text(v)
        if k and k not in seen:
            seen.add(k)
            values.append(v.strip())
    return values[:MAX_OPTIONS]


def history_entry(action, nodes, text=''):
    """Aktionshistorie exakt im NNetNav-Format (get_action_description_bgym)."""
    k = action['kind']
    if k == 'click':
        return f"click [{action['bid']}] where [{action['bid']}] is {nodes.get(action['bid'], {}).get('name', '')}"
    if k == 'type':
        shown = text + (' ' if action['enter'] else '')   # NNetNav: '\n' für Enter wird zu ' '
        return f"type [{action['bid']}] [{shown}] where [{action['bid']}] is {nodes.get(action['bid'], {}).get('name', '')}"
    if k == 'scroll':
        return f"scroll [{action['direction']}]"
    if k == 'stop':
        return f'stop [{text}]'
    return k


def observation_text(tree, url, goal, history):
    """Der NNetNav-Benutzerturn, wörtlich: genau das sah der Demonstrator."""
    past = '\n'.join(f'{i + 1}: {a}' for i, a in enumerate(history))
    return f'OBSERVATION:\n{tree}\nURL: {url}\nOBJECTIVE: {goal}\nPREVIOUS ACTIONS:\n{past}'


def value_context(goal, url, field, history):
    past = '\n'.join(f'{i + 1}: {a}' for i, a in enumerate(history))
    return f'OBJECTIVE: {goal}\nURL: {url}\nINPUT FIELD: {field}\nPREVIOUS ACTIONS:\n{past}'


def site_of(url):
    host = urlparse(url).netloc.lower().split(':')[0].removeprefix('www.')
    parts = host.split('.')
    if len(parts) >= 3 and parts[-2] in ('co', 'com', 'ac', 'org', 'gov') and len(parts[-1]) == 2:
        return '.'.join(parts[-3:])
    return '.'.join(parts[-2:]) if len(parts) >= 2 else host

## 5 · NNetNav laden, umwandeln, nach Sites trennen
`train.jsonl` (3,2 GB, 48.339 Schritte) wird einmal vom Hub geladen (fester Commit) und zeilenweise gelesen; die Daten liegen nie
komplett im Speicher. Die richtige Aktion stammt nur aus der Demonstration; Kandidaten entstehen nur aus dem Zustand. Fehlt die
demonstrierte Aktion unter den Kandidaten, wird die Zeile gezählt und nicht verwendet.

**Split nach Website:** Alle Schritte auf `amazon.com`, `allrecipes.com` und `cambridge.org` bilden die Held-out-Menge. Aufgaben,
die eine dieser Sites auch nur berühren, fehlen komplett im Training. Innerhalb jeder Seite wird nach Zustand dedupliziert, und
kein Held-out-Baum darf im Training vorkommen.

**Für Colab:** Metadaten, Held-out-Auswahl und die gezogenen Trainingsentscheidungen werden im Laufverzeichnis (Drive) abgelegt.
Ein fortgesetzter Lauf braucht `train.jsonl` danach nicht mehr.

In [ ]:
# @title Eine NNetNav-Zeile in eine Entscheidung umwandeln
USER_RE = re.compile(r'\A\s*OBSERVATION:\s*\n(?P<tree>.*)\nURL:\s*(?P<url>[^\n]*)'
                     r'\nOBJECTIVE:\s*(?P<goal>.*?)\nPREVIOUS ACTIONS:\s*\n?(?P<history>.*)\Z', re.S)
UNSUPPORTED = ('hover', 'press', 'goto', 'new_tab', 'tab_focus', 'close_tab', 'go_forward', 'page_focus')


def raw_action(output):
    text = re.sub(r'<\|[^>]+\|>', '', output or '')
    tail = text.rsplit('the next action I will perform is', 1)[-1]
    blocks = re.findall(r'`{1,4}\s*(.*?)\s*`{1,4}', tail, re.S)
    raw = (blocks[0] if blocks else tail).strip().strip('`').strip()
    return re.sub(r'\s+where \[[^\]]+\] is .*$', '', raw, flags=re.S).rstrip('.').strip()


def parse_gold(output):
    raw = raw_action(output)
    if m := re.fullmatch(r'click\s*\[([A-Za-z0-9]+)\]', raw):
        return dict(kind='click', bid=m[1])
    if m := re.fullmatch(r'type\s*\[([A-Za-z0-9]+)\]\s*\[(.*?)\](?:\s*\[([01])\])?', raw, re.S):
        return dict(kind='type', bid=m[1], text=m[2], enter=m[3] != '0')
    if m := re.fullmatch(r'scroll\s*\[?(up|down)\]?', raw):
        return dict(kind='scroll', direction=m[1])
    if raw == 'go_back':
        return dict(kind='go_back')
    if re.fullmatch(r'stop(\s*\[.*\])?', raw, re.S):
        return dict(kind='stop')
    kind = re.match(r'[a-z_]+', raw)
    return 'unsupported:' + (kind[0] if kind and kind[0] in UNSUPPORTED else 'unparseable')


def convert_row(row):
    """Eine NNetNav-Zeile -> (Entscheidung | None, Grund). Kandidaten entstehen nur aus dem Zustand."""
    users = [m['content'] for m in row.get('messages', []) if m.get('role') == 'user']
    m = USER_RE.fullmatch(users[-1]) if users else None
    if not m:
        return None, 'other_format'
    gold = parse_gold(row.get('output'))
    if isinstance(gold, str):
        return None, gold
    tree, url, goal = m['tree'], m['url'].strip(), m['goal'].strip()
    history = [l for l in m['history'].strip().split('\n') if l.strip()]
    actions, nodes = candidates(tree)
    if gold['kind'] in ('click', 'type') and gold['bid'] not in nodes:
        return None, 'target_not_in_tree'
    if gold['kind'] == 'type':
        if not gold['text'].strip():
            return None, 'empty_type_text'
        if nodes[gold['bid']]['role'] not in TYPE_ROLES:
            return None, 'type_into_non_input'
    if len(actions) > MAX_OPTIONS:
        return None, 'too_many_actions'
    keys = [action_key(a) for a in actions]
    if action_key(gold) not in keys:
        return None, f"not_offered:{gold['kind']}"
    rec = dict(task=str(row.get('task_name') or row.get('id')), site=site_of(url), goal=goal, url=url,
               user_text=users[-1].strip(), history=history, actions=actions, gold=keys.index(action_key(gold)),
               kind=gold['kind'])
    if gold['kind'] == 'type':
        n = nodes[gold['bid']]
        rec['field'] = f"[{gold['bid']}] {n['role']} {short(n['name'])!r}"
        rec['values'] = value_candidates(goal, history)
        target = norm_text(gold['text'])
        rec['value_gold'] = next((i for i, v in enumerate(rec['values']) if norm_text(v) == target), -1)
    rec['state_hash'] = hashlib.sha1(rec['user_text'].encode()).hexdigest()
    rec['tree_hash'] = hashlib.sha1(m['tree'].encode()).hexdigest()
    return rec, None

In [ ]:
# @title Kompletten Split einlesen (einmalig), Metadaten je Zeile
COMPARE_URLS = ['https://arxiv.org/', 'https://huggingface.co/', 'https://www.wolframalpha.com/', 'https://www.coursera.org/']
meta_file, scan_file = NN_DIR / 'meta.jsonl', NN_DIR / 'scan.json'
_RAW = None


def raw_jsonl():
    # train.jsonl erst laden, wenn es gebraucht wird (HF-Cache; mit NNETNAV_ON_DRIVE auf Drive).
    global _RAW
    if _RAW is None:
        t = time.time()
        kw = {'cache_dir': str(ROOT / 'hf-cache')} if NNETNAV_ON_DRIVE else {}
        _RAW = hf_hub_download(DATASET_ID, 'train.jsonl', repo_type='dataset', revision=DATASET_SHA, **kw)
        TIMINGS['download_s'] = time.time() - t
        print(f'train.jsonl bereit: {os.path.getsize(_RAW) / 2**30:.1f} GiB in {time.time() - t:.0f} s')
    return _RAW


t0 = time.time()
if meta_file.exists() and scan_file.exists():
    meta = [json.loads(l) for l in meta_file.open()]
    scan = json.loads(scan_file.read_text())
else:
    meta, reasons, gold_kinds, cand_counts, compare_trees = [], collections.Counter(), collections.Counter(), [], {}
    task_sites, prop_names = collections.defaultdict(set), collections.Counter()
    line_no = 0
    path = raw_jsonl()
    with open(path, 'rb') as f, tqdm(total=os.path.getsize(path), unit='B', unit_scale=True, desc='NNetNav', mininterval=5) as bar:
        while line := f.readline():
            offset = f.tell() - len(line)
            line_no += 1
            bar.update(len(line))
            row = json.loads(line)
            users = [m['content'] for m in row['messages'] if m['role'] == 'user']
            um = USER_RE.fullmatch(users[-1]) if users else None
            if um:
                task_sites[str(row.get('task_name'))].add(site_of(um['url'].strip()))
                g = parse_gold(row.get('output'))
                gold_kinds[g['kind'] if isinstance(g, dict) else g] += 1
                if line_no % 25 == 0:
                    prop_names.update(re.findall(r", ([a-zA-Z]+)=", um['tree']))
                if um['url'].strip() in COMPARE_URLS and um['history'].strip() == '1: None':
                    compare_trees.setdefault(um['url'].strip(), um['tree'])
            rec, why = convert_row(row)
            reasons[why or 'kept'] += 1
            if rec:
                cand_counts.append(len(rec['actions']))
                meta.append(dict(offset=offset, task=rec['task'], site=rec['site'], kind=rec['kind'],
                                 state=rec['state_hash'], tree=rec['tree_hash'], value_gold=rec.get('value_gold')))
    scan = dict(rows=line_no, reasons=dict(reasons), gold_kinds=dict(gold_kinds), cand_counts=cand_counts,
                task_sites={t: sorted(s) for t, s in task_sites.items()}, compare_trees=compare_trees,
                prop_names=dict(prop_names))
    with meta_file.open('w') as f:
        for m in meta:
            f.write(json.dumps(m) + '\n')
    scan_file.write_text(json.dumps(scan))
TIMINGS['scan_s'] = time.time() - t0

reasons = pd.Series(scan['reasons']).sort_values(ascending=False)
print(f"{scan['rows']} Zeilen gelesen, {reasons['kept']} verwendbar ({reasons['kept'] / scan['rows']:.1%}).")
display(reasons.rename('Zeilen').to_frame())
kept_by_kind = collections.Counter(m['kind'] for m in meta)
display(pd.DataFrame([{'Aktion': k, 'demonstriert': scan['gold_kinds'].get(k, 0), 'verwendbar': kept_by_kind[k],
                       'Anteil': kept_by_kind[k] / max(1, scan['gold_kinds'].get(k, 0))}
                      for k in ['click', 'type', 'stop', 'scroll', 'go_back']]).style.format({'Anteil': '{:.1%}'}))
cc = sorted(scan['cand_counts'])
type_meta = [m for m in meta if m['kind'] == 'type']
print(f"Kandidaten je Zustand: Median {cc[len(cc) // 2]}, p90 {cc[int(len(cc) * .9)]}, Max {cc[-1]}")
print(f"Texteingaben mit demonstriertem Text unter den Wertkandidaten: "
      f"{sum(m['value_gold'] >= 0 for m in type_meta)}/{len(type_meta)} "
      f"({sum(m['value_gold'] >= 0 for m in type_meta) / len(type_meta):.1%})")

## 5b · Aufträge prüfen: passt der Auftrag zur aufgezeichneten Trajektorie?
NNetNav-Aufträge wurden **nachträglich** von einem LLM zu Explorations-Trajektorien geschrieben. Manche sind unbrauchbar –
etwa *„Book a flight from Wednesday to Monday, January 1 to 6 does not fit this situation, …“* – oder beschreiben etwas,
das die Aufzeichnung nie versucht. Solche Beispiele bringen dem Modell falsche Zuordnungen bei.

Claude Haiku 4.5 hat jede der 5.017 Aufgaben einmal beurteilt (distilabel-Pipeline, strukturierte Ausgabe `keep`/`drop` mit
Begründung; nur Auftrag, Aktionen mit Elementnamen und URLs, keine Bäume; 6,91 $ für alle Aufgaben). Ergebnisse, Antwort-Cache und
Kostenbuch liegen öffentlich im Datensatz-Repo und werden von dort geladen (fester Commit), **ohne API-Aufruf**.

Neu erzeugen nur mit `ALLOW_PAID_API = True`, Secret `ANTHROPIC_API_KEY` und `RUN_FILTER = True`: kleine Stichprobe, harte
Kostengrenze. Bereits bezahlte Antworten kommen vorher aus dem Hub-Cache (Wiederholungen kosten nichts); neue Antworten werden
nach `FILTER_SYNC_REPO` hochgeladen, wenn der Token dort Schreibrecht hat, sonst bleiben sie lokal.
Gefiltert wird nur das **Training**. Die Held-out-Menge bleibt unverändert.

In [ ]:
# @title Auftragsfilter: Ergebnisse vom Hub (optional neu erzeugen)
RUN_FILTER = False        # @param {type:'boolean'}   nur mit ALLOW_PAID_API
FILTER_TASKS = 50         # @param {type:'integer'}   Stichprobe für die Neuerzeugung, 0 = alle 5.017 Aufgaben (≈ 6,90 $)
FILTER_BUDGET_USD = 1.0   # @param {type:'number'}    harte Obergrenze für neue Kosten dieses Laufs
TOTAL_BUDGET_USD = 95.0   # @param {type:'number'}    harte Obergrenze für das gesamte Kostenbuch des Filters
FILTER_SYNC_REPO = ''     # @param {type:'string'}    leer = FILTER_REPO (Upload nur mit Schreibrecht)
import objective_filter as of

VERDICTS = of.load_verdicts(FILTER_REPO, HF_TOKEN, FILTER_REVISION)
vdf = pd.DataFrame(VERDICTS.values())
display(pd.crosstab(vdf.split, vdf.decision, margins=True))
print('Modell:', vdf.model.iloc[0], '| Repo-Commit:', FILTER_REVISION[:10])
for r in vdf[vdf.decision == 'drop'].sample(5, random_state=1).itertuples():
    print(f'DROP  {short(r.objective, 110)}\n      -> {r.reason}')

if RUN_FILTER and not ALLOW_PAID_API:
    print('RUN_FILTER ignoriert: ALLOW_PAID_API ist aus.')
elif RUN_FILTER:
    FILTER_DIR = ROOT / 'objective-filter'
    FILTER_DIR.mkdir(parents=True, exist_ok=True)
    sync_repo = FILTER_SYNC_REPO or FILTER_REPO
    for repo in dict.fromkeys([FILTER_REPO, sync_repo]):
        print('Hub-Cache eingespielt aus', repo, ':', of.restore_caches(repo, FILTER_DIR, HF_TOKEN))
    summaries = of.task_summaries(of.scan_steps(raw_jsonl()))
    random.Random(0).shuffle(summaries)
    chosen = summaries[:FILTER_TASKS] if FILTER_TASKS else summaries
    cached = of.cached_verdicts(chosen, FILTER_DIR / 'llm_cache.sqlite')
    print(f'{len(chosen)} Aufgaben, {len(cached)} schon bezahlt (Cache); Schätzung für den Rest '
          f'${of.estimate_cost([s for s in chosen if s["task"] not in cached]):.2f}, Obergrenze ${FILTER_BUDGET_USD:.2f}')
    t0 = time.time()
    pipeline = of.build_pipeline(chosen, FILTER_DIR, run_cap_usd=FILTER_BUDGET_USD, total_cap_usd=TOTAL_BUDGET_USD)
    try:
        distiset = pipeline.run(use_cache=False)
    finally:
        new_cost = sum(of.usage_cost(r['model'], r['usage']) for r in of.read_jsonl(FILTER_DIR / 'api_ledger.jsonl') if r['t'] >= t0)
        if new_cost > 0:   # auch bei Budgetstopp: Bezahltes sofort sichern
            url = of.push_caches(sync_repo, FILTER_DIR, HF_TOKEN, f'objective filter rerun: {len(chosen)} tasks, ${new_cost:.2f}') \
                if of.hub_can_write(sync_repo, HF_TOKEN) else None
            print('Bezahlte Antworten hochgeladen:', url or f'kein Schreibrecht auf {sync_repo}, nur lokal in {FILTER_DIR}')
    fresh = of.collect(distiset)
    agree = [VERDICTS[t]['decision'] == (r['verdict'] or {}).get('decision') for t, r in fresh.items() if t in VERDICTS]
    print(f'Neue Kosten ${new_cost:.2f}; Übereinstimmung mit dem veröffentlichten Filter: {sum(agree)}/{len(agree)}')


def task_kept(task):
    # Aufgaben ohne Urteil (sollte nicht vorkommen) werden behalten.
    v = VERDICTS.get(task)
    return v is None or v['decision'] != 'drop'

## 6 · Entscheidungen als Decider-Prompts
Der Kontext ist der NNetNav-Benutzerturn, wörtlich – `OBSERVATION`, `URL`, `OBJECTIVE`, `PREVIOUS ACTIONS`. Danach folgen Frage
und Optionen in Deciders nativem `state_first`-Format; der Kontext wird nie gekürzt. Das Modell liest am Antwortslot die Logits der
Label-Token aus: ein Forward Pass pro Entscheidung.

**Tempo ohne Qualitätsverlust.** Batches werden rechts aufgefüllt, und das Modell ist kausal: Kein echtes Token sieht je ein
Füll-Token. Die Attention-Maske ändert an den Antwortslots also nichts – ohne sie darf PyTorchs SDPA aber einen schnellen
Kernel nehmen (Flash ab Ampere, auf der T4 den speichersparenden Kernel statt der quadratischen Referenz). Gemessen im vollen
Lauf auf einer A6000: Training **1,80×**, Inferenz **1,42×** schneller; die Logits weichen nur im BF16-Rauschen ab. Aus demselben
Grund darf rechts beliebig aufgefüllt werden: Wir runden jede Länge auf 512 Tokens auf, damit nur wenige Sequenzlängen
formspezifische Kernel erzeugen (sonst wächst der Hauptspeicher mit jeder neuen Länge).

In [ ]:
# @title Prompt, Auswahl-Logits, Trainer
ACTION_QUESTION = ('Which next browser action best advances the objective, given the page, the URL and the previous '
                   'actions? Choose exactly one of the offered actions.')
VALUE_QUESTION = 'Which text should be typed into the input field to advance the objective?'
EXTRACT_QUESTION = 'Which text on the page answers this question: {q}'
KINDS = ['click', 'type', 'stop', 'scroll', 'go_back']


def make_item(tokenizer, context, question, options, gold=-1, rng=None):
    """Deciders nativer state_first-Prompt; der Kontext wird nie gekürzt."""
    example = SimpleNamespace(context=context, qs=[SimpleNamespace(text=question, options=options, gold=gold)])
    n_ctx = len(tokenizer.encode('Context:\n' + context, add_special_tokens=False))
    return build_decision_prompt(example, tokenizer, rng=rng or random.Random(0), max_options=MAX_OPTIONS,
                                 max_ctx_tokens=n_ctx + 8, layout='state_first')


def action_item(tokenizer, rec, rng=None):
    return make_item(tokenizer, rec['user_text'], ACTION_QUESTION, [a['label'] for a in rec['actions']], rec['gold'], rng)


def value_item(tokenizer, rec, rng=None):
    ctx = value_context(rec['goal'], rec['url'], rec['field'], rec['history'])
    return make_item(tokenizer, ctx, VALUE_QUESTION, rec['values'], rec['value_gold'], rng)


def choice_logits(peft_model, batch, letter_token_ids):
    """Wie decider.model.DecisionModel.slot_logits: Hidden State am Antwortslot x Buchstaben-Embeddings.
    Ohne Attention-Maske: rechts aufgefüllt + kausal = identische Slots, aber SDPA kann den Flash-Kernel nehmen."""
    lm = peft_model.get_base_model() if hasattr(peft_model, 'get_base_model') else peft_model
    hidden = lm.model(input_ids=batch['input_ids'], attention_mask=None, use_cache=False, return_dict=True).last_hidden_state
    slots = hidden[batch['slot_batch'], batch['slot_idx']]
    weights = lm.lm_head.weight.index_select(0, letter_token_ids)
    scores = F.linear(slots, weights).float()
    valid = torch.arange(len(letter_token_ids), device=scores.device)[None, :] < batch['nopts'][:, None]
    return scores.masked_fill(~valid, float('-inf'))


class DecisionTrainer(Trainer):
    def __init__(self, *args, letter_token_ids=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.letter_token_ids = letter_token_ids

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        scores = choice_logits(model, inputs, self.letter_token_ids)
        loss = F.cross_entropy(scores, inputs['labels'], reduction='none')
        if 'weights' in inputs:                    # optionale Gewichte je Aktionstyp (CFG['kind_weights'])
            loss = loss * inputs['weights'].to(loss.device)
        loss = loss.mean()
        return (loss, {'logits': scores}) if return_outputs else loss


LENGTH_BUCKET = 512


def pad_to_bucket(batch, pad_id, multiple=LENGTH_BUCKET):
    # Rechts auf ein Vielfaches von 512 Tokens auffüllen (decider rundet nur auf 64): Jede neue Sequenzlänge kostet
    # formspezifische Kernel (Triton-Autotuning der Gated-DeltaNet-Schichten) und bleibt im Hauptspeicher, gemessen
    # rund 17 MB je Länge. Mit 512er-Stufen bleibt das unter 1 GB – wichtig bei 12,7 GB RAM auf Colab.
    # Kausales Modell ohne Maske: Füll-Tokens am Ende ändern die Antwortslots nicht.
    extra = -batch['input_ids'].shape[1] % multiple
    if extra:
        batch['input_ids'] = F.pad(batch['input_ids'], (0, extra), value=pad_id)
        batch['attention_mask'] = F.pad(batch['attention_mask'], (0, extra), value=0)
    return batch


def decision_collator(pad_id):
    def collate(items):
        batch = pad_to_bucket(collate_decisions(items, pad_id), pad_id)
        batch['labels'] = batch.pop('golds')
        if all('weight' in it for it in items):
            batch['weights'] = torch.tensor([it['weight'] for it in items])
        return {k: v for k, v in batch.items() if torch.is_tensor(v)}
    return collate


def with_oom_retry(fn, *args, tries=30, **kwargs):
    """Die GPU kann geteilt sein: bei OOM Speicher freigeben, warten, erneut versuchen."""
    for attempt in range(tries):
        try:
            return fn(*args, **kwargs)
        except torch.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache(); time.sleep(min(300, 20 * (attempt + 1)))
    return fn(*args, **kwargs)


@contextmanager
def active_adapter(model, adapter):
    """adapter: False = Basismodell, True = eigener Adapter ('default'), str = weiterer geladener Adapter."""
    if adapter is False:
        with (model.disable_adapter() if hasattr(model, 'disable_adapter') else nullcontext()):
            yield
        return
    name = 'default' if adapter is True else adapter
    before = model.active_adapter
    model.set_adapter(name)
    try:
        yield
    finally:
        model.set_adapter(before)


@torch.inference_mode()
def score_options(model, tokenizer, letter_token_ids, context, question, options, adapter=True, seed=0):
    """Wahrscheinlichkeit je Option in Originalreihenfolge (Optionen werden wie im Training permutiert)."""
    model.eval()
    item = make_item(tokenizer, context, question, options, -1, random.Random(seed))
    batch = pad_to_bucket(collate_decisions([item], tokenizer.pad_token_id), tokenizer.pad_token_id)
    batch = {k: v.to(letter_token_ids.device) for k, v in batch.items() if torch.is_tensor(v)}
    with active_adapter(model, adapter) if hasattr(model, 'set_adapter') else nullcontext():
        scores = choice_logits(model, batch, letter_token_ids)[0, :item['nopts'][0]]
    probs = torch.softmax(scores, -1).cpu()
    out = [0.0] * len(options)
    for pos, orig in enumerate(item['perms'][0]):
        out[orig] = float(probs[pos])
    return out

In [ ]:
# @title Held-out-Auswahl (wie im vollen Lauf), Validierung, Trainingsstichprobe (gecacht)
def read_record(f, m):
    # Eine Entscheidung aus train.jsonl über ihren Byte-Offset.
    f.seek(m['offset'])
    rec, why = convert_row(json.loads(f.readline()))
    assert why is None and rec['state_hash'] == m['state']
    return rec


def load_records(metas):
    with open(raw_jsonl(), 'rb') as f:
        return [read_record(f, m) for m in metas]


def token_count(rec):
    return len(action_item(TOKENIZER, rec)['ids'])


def dedupe(metas):
    seen, out = set(), []
    for m in metas:
        if m['state'] not in seen:
            seen.add(m['state'])
            out.append(m)
    return out


def shuffled(metas, rng):
    order = metas[:]
    rng.shuffle(order)
    return order


def pick(order, n, per_task, max_tokens):
    # In fester Reihenfolge, höchstens per_task Schritte je Aufgabe; zu lange Zustände werden gezählt und übersprungen.
    picked, per, too_long = [], collections.Counter(), 0
    with open(raw_jsonl(), 'rb') as f:
        for m in order:
            if n is not None and len(picked) >= n:
                break
            if per_task and per[m['task']] >= per_task:
                continue
            rec = read_record(f, m)
            rec['n_tokens'] = token_count(rec)
            if rec['n_tokens'] > max_tokens:
                too_long += 1
                continue
            per[m['task']] += 1
            picked.append(rec)
    return picked, too_long


def write_jsonl(path, records):
    tmp = Path(str(path) + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    tmp.replace(path)                      # erst vollständig schreiben, dann umbenennen (Abbruch-sicher auf Drive)


def read_jsonl(path):
    return [json.loads(l) for l in Path(path).open(encoding='utf-8')]


def digest(records):
    return hashlib.sha256(''.join(r['state_hash'] for r in records).encode()).hexdigest()[:16]


t0 = time.time()
held_tasks = {t for t, s in scan['task_sites'].items() if set(s) & HELDOUT_SITES}
train_all = [m for m in meta if m['task'] not in held_tasks]
eval_all = [m for m in meta if m['site'] in HELDOUT_SITES]
train_pool, eval_pool_all = dedupe(train_all), dedupe(eval_all)
train_trees = {m['tree'] for m in train_pool}
eval_pool = [m for m in eval_pool_all if m['tree'] not in train_trees]
# Derselbe Zufallsstrom wie im vollen Lauf (und in v1): erst den Trainingspool mischen, dann die Held-out-Reihenfolge,
# dann Texteingaben. Nur die Reihenfolgen werden berechnet; gelesen wird nur, was noch nicht im Cache liegt.
rng = random.Random(SEED)
rng.shuffle(train_pool[:])
eval_order = shuffled(eval_pool, rng)
tv = [m for m in train_pool if m['kind'] == 'type' and m['value_gold'] >= 0]
ev = [m for m in eval_pool if m['kind'] == 'type' and m['value_gold'] >= 0]
rng.shuffle(tv); rng.shuffle(ev)
eval_file, eval_values_file = NN_DIR / 'eval.jsonl', NN_DIR / 'eval_values.jsonl'
if eval_file.exists() and eval_values_file.exists():
    eval_all_records, eval_values = read_jsonl(eval_file), read_jsonl(eval_values_file)
else:
    eval_all_records, eval_too_long = pick(eval_order, N_EVAL_ALL, 2, EVAL_MAX_TOKENS)
    eval_values = load_records(ev[:N_VALUE_EVAL])
    write_jsonl(eval_file, eval_all_records); write_jsonl(eval_values_file, eval_values)
HELDOUT_DIGEST = [digest(eval_all_records), digest(eval_values)]
assert HELDOUT_DIGEST == PUBLISHED_CONTRACT['heldout_digest'], \
    f'Held-out-Auswahl {HELDOUT_DIGEST} weicht vom veröffentlichten Lauf {PUBLISHED_CONTRACT["heldout_digest"]} ab'
eval_records = eval_all_records[:CFG['n_eval']]          # dieselben ersten n_eval in jedem Lauf

# Validierung: ganze Aufgaben von Trainings-Sites, nie trainiert; sie wählt den Checkpoint, Held-out sieht dabei nichts.
tasks = sorted({m['task'] for m in train_pool})
random.Random(SEED + 7).shuffle(tasks)
val_tasks = set(tasks[:CFG['n_val_tasks']])
DATA_DIR = RUN_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
files = {k: DATA_DIR / f'{k}.jsonl' for k in ['val', 'val_values', 'train_actions', 'train_values']}
if all(p.exists() for p in files.values()):
    val_records, val_values = read_jsonl(files['val']), read_jsonl(files['val_values'])
    train_actions, train_values = read_jsonl(files['train_actions']), read_jsonl(files['train_values'])
    selection = json.loads((DATA_DIR / 'selection.json').read_text())
else:
    val_records, _ = pick(shuffled([m for m in train_pool if m['task'] in val_tasks], random.Random(SEED + 8)),
                          CFG['n_val'], 2, EVAL_MAX_TOKENS)
    val_values = load_records([m for m in tv if m['task'] in val_tasks][:CFG['n_value_val']])
    # Training: Trainings-Sites ohne Validierungsaufgaben, nach Auftragsfilter; zufällige Stichprobe im Tokenbudget.
    fit_pool = [m for m in train_pool if m['task'] not in val_tasks]
    fit = [m for m in fit_pool if task_kept(m['task'])] if CFG['objective_filter'] else fit_pool
    rng_train = random.Random(SEED)
    value_pool = [m for m in fit if m['kind'] == 'type' and m['value_gold'] >= 0]
    rng_train.shuffle(value_pool)
    train_order = shuffled(fit, rng_train) if CFG['n_train'] else fit
    train_actions, n_too_long = pick(tqdm(train_order, desc='Trainingsstichprobe', mininterval=10), CFG['n_train'],
                                     CFG['per_task'], CFG['max_tokens'])
    train_values = load_records(value_pool[:CFG['n_value_train']] if CFG['n_value_train'] else value_pool)
    for r in train_values:
        r['value_tokens'] = len(value_item(TOKENIZER, r)['ids'])
    selection = dict(fit_pool=len(fit_pool), after_filter=len(fit), removed_by_filter=len(fit_pool) - len(fit),
                     over_token_budget_skipped=n_too_long, train_actions=len(train_actions), train_values=len(train_values))
    for k, v in [('val', val_records), ('val_values', val_values), ('train_actions', train_actions), ('train_values', train_values)]:
        write_jsonl(files[k], v)
    (DATA_DIR / 'selection.json').write_text(json.dumps(selection, indent=2))
TIMINGS['split_s'] = time.time() - t0

assert not {r['site'] for r in train_actions} & HELDOUT_SITES
assert not {r['task'] for r in train_actions + train_values} & ({r['task'] for r in eval_all_records} | val_tasks)
assert not {r['tree_hash'] for r in train_actions} & {r['tree_hash'] for r in eval_all_records}
split_stats = dict(train_pool_rows=len(train_pool), train_pool_tasks=len({m['task'] for m in train_pool}),
                   train_sites=len({m['site'] for m in train_pool}), eval_pool_rows=len(eval_pool),
                   eval_pool_tasks=len({m['task'] for m in eval_pool}), val_tasks=len(val_tasks), **selection,
                   heldout_digest=' / '.join(HELDOUT_DIGEST))
display(pd.Series(split_stats, name='Split').to_frame())
display(pd.DataFrame({
    'Training': pd.Series(collections.Counter(r['kind'] for r in train_actions)),
    'Validierung': pd.Series(collections.Counter(r['kind'] for r in val_records)),
    f'Held-out (ausgewertet, n={len(eval_records)})': pd.Series(collections.Counter(r['kind'] for r in eval_records))}).fillna(0).astype(int))
tok = pd.Series([r['n_tokens'] for r in train_actions])
TRAIN_TOKENS = int(tok.sum() + sum(r['value_tokens'] for r in train_values))
print(f'Trainingsentscheidungen: {len(train_actions)} Aktionen (bis {CFG["max_tokens"]} Tokens), {len(train_values)} Texteingaben')
print(f'Tokens je Entscheidung: Median {tok.median():.0f}, p90 {tok.quantile(.9):.0f}, Max {tok.max()} | Summe {TRAIN_TOKENS / 1e6:.1f} M')
print('Trainingssites:', len({r['site'] for r in train_actions}), collections.Counter(r['site'] for r in train_actions).most_common(8))
print('Held-out-Menge wie im vollen Lauf:', HELDOUT_DIGEST, f'(Fingerabdruck geprüft; ausgewertet die ersten {len(eval_records)})')

In [ ]:
# @title Ein echtes Trainingsbeispiel
ex = next(r for r in train_actions if r['kind'] == 'click' and len(r['history']) > 2)
vx = next((r for r in train_values if len(r['values']) > 5), train_values[0])
print('AUFTRAG:', ex['goal'], '\nURL:', ex['url'], '\nHISTORIE:', *ex['history'], sep='\n')
print('\nBAUM (Ausschnitt):\n' + '\n'.join(ex['user_text'].split('\n')[1:30]))
print(f"\nRICHTIG: {ex['actions'][ex['gold']]['label']}   ({len(ex['actions'])} Kandidaten, {ex['n_tokens']} Tokens)")
print('\nTEXTEINGABE:', vx['goal'], '| Feld:', vx['field'])
print('Wertkandidaten:', vx['values'][:12], '...', '\nRICHTIG:', repr(vx['values'][vx['value_gold']]))

### Stimmt das Live-Format mit dem Trainingsformat überein?
Vier Startseiten aus NNetNav öffnen wir jetzt live und vergleichen mit dem aufgezeichneten Baum derselben URL:
Formatmerkmale (Tabs, IDs, Flags, Attribute) und den Anteil gemeinsamer `(Rolle, Name)`-Zeilen.
Websites ändern sich seit der Aufzeichnung (Dezember 2024); identische Inhalte sind nicht zu erwarten,
identisches Format schon.

In [ ]:
# @title Live-Baum vs. Trainingsbaum
def tree_profile(tree):
    lines = tree.split('\n')
    el = [l for l in lines if NODE_RE.match(l)]
    links = [l for l in el if NODE_RE.match(l)[3] == 'link']
    statics = [l for l in lines if l.lstrip('\t').startswith('StaticText')]
    return {
        'Zeilen': len(lines),
        'Einrückung nur Tabs': all(not l.startswith(' ') for l in lines),
        'Zeilen mit [bid]': len(el) / len(lines),
        'StaticText ohne bid': sum(not re.match(r'^\t*\[', l) for l in statics) / max(1, len(statics)),
        'Elemente clickable': sum(', clickable' in l for l in el) / max(1, len(el)),
        'Elemente visible': sum(', visible' in l for l in el) / max(1, len(el)),
        'Links mit url=': sum("url='" in l for l in links) / max(1, len(links)),
        'InlineTextBox': sum('InlineTextBox' in l for l in lines),
        'JSON-Zitate ("...")': sum(bool(re.match(r'^\t*(\[\w+\] )?\w+ "', l)) for l in lines) / len(lines),
    }


def role_name_lines(tree):
    return collections.Counter(re.sub(r'^\t*(\[[A-Za-z0-9]+\] )?', '', l).split(', ')[0] for l in tree.split('\n'))


async def capture(urls):
    trees = {}
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(viewport={'width': 1280, 'height': 720})
        page = await ctx.new_page()
        for u in urls:
            try:
                await page.goto(u, wait_until='load', timeout=45000)
                await page.wait_for_timeout(1500)
                trees[u] = (await bgym_observe(page, BGYM_JS))[0]
            except Exception as exc:
                print('nicht erreichbar:', u, type(exc).__name__)
        await browser.close()
    return trees

live_trees = await capture(list(scan['compare_trees']))
rows = []
for u, live in live_trees.items():
    rec = scan['compare_trees'][u]
    a, b = role_name_lines(rec), role_name_lines(live)
    shared = sum((a & b).values()) / max(1, sum(a.values()))
    rows.append({'URL': u, 'Quelle': 'NNetNav (Dez. 2024)', **tree_profile(rec)})
    rows.append({'URL': u, 'Quelle': 'live, dieses Notebook', **tree_profile(live), 'gemeinsame Zeilen': shared})
display(pd.DataFrame(rows).set_index(['URL', 'Quelle']).T.round(3))
live_props = set().union(*[re.findall(r", ([a-zA-Z]+)=", t) for t in live_trees.values()])
print('Attributnamen live, die in den NNetNav-Bäumen nie vorkommen:', sorted(live_props - set(scan['prop_names'])) or 'keine')
first = next(iter(live_trees))
print('\nNNetNav:\n' + '\n'.join(scan['compare_trees'][first].split('\n')[:12]))
print('\nLive:\n' + '\n'.join(live_trees[first].split('\n')[:12]))

## 7 · Basismodell und LoRA
LoRA-Rang 16 auf allen linearen Schichten des hybriden Backbones (Gated DeltaNet + Attention). Kein neuer
Klassifikationskopf: Die Auswahl-Logits kommen aus den vorhandenen Label-Token-Embeddings, genau wie in Deciders eigener
Inferenz. Gewichte in der gewählten Präzision (BF16 bzw. FP16 auf der T4), LoRA-Parameter in FP32.

In [ ]:
# @title Decider laden, LoRA aktivieren
def load_base():
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_SHA, dtype=DTYPE, attn_implementation='sdpa')
    base.config.use_cache = False
    base.config.pad_token_id = TOKENIZER.pad_token_id
    return base


LETTERS = torch.tensor(letter_ids(TOKENIZER), dtype=torch.long, device=DEVICE)
adapter_ready = (ADAPTER_DIR / 'adapter_model.safetensors').exists()
if adapter_ready:
    print('Fertiger Adapter gefunden – Training wird übersprungen:', ADAPTER_DIR)
else:
    wait_for_gpu(8)
    model = get_peft_model(load_base(), LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=CFG['rank'], lora_alpha=2 * CFG['rank'], lora_dropout=0.0,
        bias='none', target_modules='all-linear'))
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.enable_input_require_grads()
    model.to(DEVICE)
    model.print_trainable_parameters()
    print('Datentypen der trainierbaren Parameter:', {str(p.dtype) for p in model.parameters() if p.requires_grad})

## 8 · Training
Alle gezogenen Aktions- und Texteingabe-Entscheidungen, zufällig gemischt, Microbatch 1, Gradient Accumulation 8,
Cosine-Schedule, eine Epoche – dasselbe Rezept wie der veröffentlichte Adapter, nur mit weniger Daten. Gradient Checkpointing
bleibt an. Alle `eval_every` Schritte misst ein Callback die Schrittgenauigkeit auf der Validierungsmenge; am Ende wird der
beste Stand zum Adapter. Checkpoints alle `save_every` Schritte im Laufverzeichnis (Drive); ein erneuter Start setzt dort fort.

Die Zeitschätzung vor dem Start ist eine grobe Annahme je GPU-Klasse; nach den ersten Schritten wird die tatsächlich
gemessene Restzeit ausgegeben.

In [ ]:
# @title Trainieren (mit Validierung, Checkpoint-Auswahl, Restzeit und Fortsetzung)
class Items(torch.utils.data.Dataset):
    # Aktions- und Texteingabe-Entscheidungen; die Optionsreihenfolge wird je Beispiel neu gemischt.
    def __init__(self, entries):
        self.entries = entries
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        rec, is_value = self.entries[i]
        rng = random.Random(SEED * 100003 + i)
        item = value_item(TOKENIZER, rec, rng) if is_value else action_item(TOKENIZER, rec, rng)
        item['weight'] = 1.0
        return item


def train_entries(actions, values):
    entries = [(r, False) for r in actions] + [(r, True) for r in values]
    random.Random(SEED).shuffle(entries)
    return entries


def evaluate_val(model):
    rows = []
    for r in val_records:
        p = with_oom_retry(score_options, model, TOKENIZER, LETTERS, r['user_text'], ACTION_QUESTION, [a['label'] for a in r['actions']])
        pred = max(range(len(p)), key=p.__getitem__)
        rows.append(dict(kind=r['kind'], correct=pred == r['gold'], pred_kind=r['actions'][pred]['kind']))
    vok = [max(range(len(p)), key=p.__getitem__) == r['value_gold'] for r in val_values
           for p in [with_oom_retry(score_options, model, TOKENIZER, LETTERS,
                                    value_context(r['goal'], r['url'], r['field'], r['history']), VALUE_QUESTION, r['values'])]]
    d = pd.DataFrame(rows)
    return dict(acc=d.correct.mean(), value_acc=sum(vok) / max(1, len(vok)),
                **{k: d[d.kind == k].correct.mean() for k in KINDS}, pred_kinds=d.pred_kind.value_counts().to_dict())


VAL_ADAPTERS = RUN_DIR / 'val_adapters'


class ValidationCallback(TrainerCallback):
    # Misst alle `every` Schritte die Validierung und legt die Adaptergewichte dieses Stands ab (für die Auswahl).
    def __init__(self, model, every, log_file):
        self.model, self.every, self.log_file = model, every, log_file
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0 or state.global_step == state.max_steps:
            t = time.time()
            res = dict(step=state.global_step, epoch=round(state.epoch, 4), **evaluate_val(self.model))
            self.model.train()
            res['seconds'] = round(time.time() - t, 1)
            self.model.save_pretrained(VAL_ADAPTERS / f'step-{state.global_step}')
            with open(self.log_file, 'a') as f:
                f.write(json.dumps(res) + '\n')
            print(f"Validierung Schritt {res['step']}: {res['acc']:.1%} (Texte {res['value_acc']:.1%})", flush=True)


class RemainingTime(TrainerCallback):
    # Gemessene Restzeit (inklusive Validierungen) nach den ersten Schritten dieses Starts.
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0, self.s0 = time.time(), state.global_step
    def on_step_end(self, args, state, control, **kwargs):
        done = state.global_step - self.s0
        if done in (3, 10) or (done > 0 and state.global_step % 50 == 0):
            rate = (time.time() - self.t0) / done
            print(f'Schritt {state.global_step}/{state.max_steps}: {rate:.1f} s/Schritt gemessen, '
                  f'Rest ≈ {(state.max_steps - state.global_step) * rate / 60:.0f} min', flush=True)


def make_trainer(model, dataset, out_dir, lr, epochs, save_steps=50, callbacks=()):
    args = TrainingArguments(
        output_dir=str(out_dir), learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=1, gradient_accumulation_steps=CFG['accumulation'],
        bf16=PRECISION == 'bf16', fp16=PRECISION == 'fp16', gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        warmup_steps=0.05, weight_decay=0.0, max_grad_norm=1.0, lr_scheduler_type='cosine',
        logging_steps=10, logging_first_step=True, save_strategy='steps', save_steps=save_steps, save_total_limit=2,
        report_to='none', remove_unused_columns=False, label_names=['labels'], dataloader_num_workers=2,
        seed=SEED, data_seed=SEED)
    trainer = DecisionTrainer(model=model, args=args, train_dataset=dataset, data_collator=decision_collator(TOKENIZER.pad_token_id),
                              processing_class=TOKENIZER, letter_token_ids=LETTERS, callbacks=list(callbacks))
    trainer.model_accepts_loss_kwargs = False      # Loss mittelt über Entscheidungen
    return trainer


def read_val_log():
    rows = {}
    for line in (RUN_DIR / 'val_log.jsonl').read_text().splitlines():
        r = json.loads(line)
        rows[r['step']] = r                         # nach einer Fortsetzung zählt die letzte Messung je Schritt
    return pd.DataFrame(sorted(rows.values(), key=lambda r: r['step']))


# Grobe Annahme des Durchsatzes (Tokens/s, Training mit Gradient Checkpointing), nicht auf jeder GPU gemessen:
# RTX A6000 (geteilt) gemessen ≈ 3.000 (voller Lauf, BF16); T4/L4/A100 abgeleitet.
ASSUMED_TOKENS_PER_S = {'T4': 700, 'L4': 1500, 'A100': 6000, 'H100': 10000, 'A6000': 3000}
entries = train_entries(train_actions, train_values)
order_digest = hashlib.sha256(json.dumps([(r['state_hash'], v) for r, v in entries]).encode()).hexdigest()[:16]
guess = next((v for k, v in ASSUMED_TOKENS_PER_S.items() if k in GPU_NAME), None)
print(f'{len(entries)} Trainingsbeispiele, {TRAIN_TOKENS / 1e6:.1f} M Tokens, '
      f'{math.ceil(len(entries) / CFG["accumulation"])} Optimizer-Schritte, Reihenfolge {order_digest}')
if guess:
    print(f'Grobe Schätzung für {GPU_NAME}: ≈ {TRAIN_TOKENS / guess / 3600:.1f} h Training '
          f'(+ {len(val_records) + len(val_values)} Validierungsentscheidungen alle {CFG["eval_every"]} Schritte)')
if not adapter_ready:
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    ckpt_dir = RUN_DIR / 'checkpoints'
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for attempt in range(20):          # geteilte GPU (lokal): nach OOM warten und beim letzten Checkpoint fortsetzen
        trainer = make_trainer(model, Items(entries), ckpt_dir, CFG['lr'], CFG['epochs'], save_steps=CFG['save_every'],
                               callbacks=[ValidationCallback(model, CFG['eval_every'], RUN_DIR / 'val_log.jsonl'), RemainingTime()])
        model.train()
        try:
            trainer.train(resume_from_checkpoint=True if any(ckpt_dir.glob('checkpoint-*')) else None)
            break
        except torch.OutOfMemoryError:
            print(f'OOM (Versuch {attempt + 1}) – warte 3 min, dann Fortsetzung beim letzten Checkpoint', flush=True)
            del trainer
            model.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache(); time.sleep(180)
    TIMINGS['train_s'] = time.time() - t0
    TIMINGS['train_peak_gib'] = torch.cuda.max_memory_allocated() / 2**30
    val_log = read_val_log()
    best = val_log.loc[val_log.acc.idxmax()]
    set_peft_model_state_dict(model, load_file(VAL_ADAPTERS / f'step-{int(best.step)}' / 'adapter_model.safetensors'))
    print(f"Bester Checkpoint: Schritt {int(best.step)} von {trainer.state.global_step}, Validierung {best.acc:.1%}")
    trainer.save_model(str(ADAPTER_DIR))
    TOKENIZER.save_pretrained(ADAPTER_DIR)
    cfg_file = ADAPTER_DIR / 'adapter_config.json'
    acfg = json.loads(cfg_file.read_text()); acfg['revision'] = MODEL_SHA; cfg_file.write_text(json.dumps(acfg, indent=2))
    pd.DataFrame(trainer.state.log_history).to_csv(RUN_DIR / 'training_log.csv', index=False)
    (RUN_DIR / 'training_recipe.json').write_text(json.dumps(dict(
        profile=PROFILE, precision=PRECISION, gpu=GPU_NAME, config=CFG, base_model=MODEL_ID, base_revision=MODEL_SHA,
        dataset=DATASET_ID, dataset_revision=DATASET_SHA,
        objective_filter=dict(repo=FILTER_REPO, revision=FILTER_REVISION) if CFG['objective_filter'] else None,
        heldout_sites=sorted(HELDOUT_SITES), heldout_digest=HELDOUT_DIGEST, n_eval=CFG['n_eval'],
        optimizer_steps=trainer.state.global_step, selected_step=int(best.step), selected_val_acc=float(best.acc),
        train_decisions=len(train_actions), train_value_decisions=len(train_values), order_digest=order_digest,
        train_kinds=dict(collections.Counter(r['kind'] for r in train_actions)), train_tokens=TRAIN_TOKENS,
        versions=versions, seconds=TIMINGS['train_s'], peak_gib=TIMINGS['train_peak_gib'], browsergym=BGYM_TAG), indent=2))
    print(f"{trainer.state.global_step} Optimizer-Schritte in {TIMINGS['train_s'] / 3600:.2f} h "
          f"({TRAIN_TOKENS / TIMINGS['train_s']:.0f} Tokens/s inkl. Validierung), Peak {TIMINGS['train_peak_gib']:.1f} GiB")
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
recipe = json.loads((RUN_DIR / 'training_recipe.json').read_text())
assert recipe['order_digest'] == order_digest, 'Adapter im Laufverzeichnis stammt aus einer anderen Trainingsreihenfolge'
history_df, val_log = pd.read_csv(RUN_DIR / 'training_log.csv'), read_val_log()

In [ ]:
# @title Loss- und Validierungsverlauf
import matplotlib.pyplot as plt
loss = history_df.dropna(subset=['loss'])
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 3))
ax.plot(loss['step'], loss['loss'], alpha=.4, label='Trainings-Loss')
ax.plot(loss['step'], loss['loss'].rolling(20, min_periods=1).mean(), label='gleitend (20)')
ax.set_xlabel('Optimizer-Schritt'); ax.set_ylabel('Cross-Entropy'); ax.legend()
ax2.plot(val_log.step, val_log.acc, marker='o', label='Aktion (gesamt)')
for k in ['click', 'type', 'stop']:
    ax2.plot(val_log.step, val_log[k], alpha=.6, label=k)
ax2.plot(val_log.step, val_log.value_acc, ls='--', label='Eingabetext')
ax2.axvline(recipe['selected_step'], color='grey', lw=.8)
ax2.set_xlabel('Optimizer-Schritt'); ax2.set_ylabel('Validierung'); ax2.legend(fontsize=8); fig.tight_layout(); plt.show()
display(val_log.set_index('step')[['epoch', 'acc'] + KINDS + ['value_acc']].style.format('{:.1%}', subset=['acc'] + KINDS + ['value_acc']))

## 9 · Adapter vom Datenträger laden und auf fremden Websites prüfen
Wir laden Basismodell plus gespeicherten Adapter neu, dazu als zweiten Adapter den **veröffentlichten v2**
(`johannhartmann/decider-2b-a11y-crawler-lora`, fester Commit: alle 26.969 + 2.613 Entscheidungen, 29 h). Alle drei
Bedingungen laufen auf denselben Gewichten mit demselben Code; das Basismodell über `disable_adapter()`.

Gemessen wird die **Schrittgenauigkeit**: Trifft das Modell die demonstrierte Aktion? Die Held-out-Seiten (Amazon, Allrecipes,
Cambridge Dictionary) kamen im Training nie vor. Andere Aktionen können im Einzelfall auch sinnvoll sein; die Demonstration ist die
Referenz. Die Ratequote ist der Mittelwert von 1/#Kandidaten. Zusatzsichten: Genauigkeit nur auf Aufgaben, die der Auftragsfilter
behält, und ein **gleichwertiges Ziel** (verschachtelte Klickziele, gleiches Eingabefeld).

In [ ]:
# @title Adapter laden (eigener + veröffentlichter v2)
wait_for_gpu(6)
model = PeftModel.from_pretrained(load_base(), ADAPTER_DIR, is_trainable=True).to(DEVICE)
model.load_adapter(MODEL_REPO, adapter_name='v2', revision=PUBLISHED_REVISION)
model.set_adapter('default')
model.eval()
print('Eigener Adapter:', ADAPTER_DIR, '| Vergleich v2:', MODEL_REPO, PUBLISHED_REVISION[:10])


def ancestors(tree):
    """bid -> Eltern-bid aus der Einrückung des BrowserGym-Baums."""
    parent, stack = {}, []
    for line in tree.split('\n'):
        m = re.match(r'^(\t*)\[([A-Za-z0-9]+)\] ', line)
        if not m:
            continue
        while stack and stack[-1][0] >= len(m[1]):
            stack.pop()
        parent[m[2]] = stack[-1][1] if stack else None
        stack.append((len(m[1]), m[2]))
    def chain(b):
        out = set()
        while parent.get(b) is not None:
            b = parent[b]; out.add(b)
        return out
    return chain


def equivalent(r, pred):
    """Sekundärmetrik: gleichwertiges Ziel (verschachtelte Klickziele, gleiches Eingabefeld)."""
    g, p = r['actions'][r['gold']], r['actions'][pred]
    if pred == r['gold']:
        return True
    if g['kind'] != p['kind'] or g['kind'] not in ('click', 'type'):
        return False
    if g['kind'] == 'type':
        return g['bid'] == p['bid']
    chain = ancestors(r['user_text'].split('\nURL:')[0])
    return g['bid'] in chain(p['bid']) or p['bid'] in chain(g['bid'])


def evaluate_actions(records, adapter, desc):
    rows = []
    for r in tqdm(records, desc=desc, mininterval=10):
        p = with_oom_retry(score_options, model, TOKENIZER, LETTERS, r['user_text'], ACTION_QUESTION,
                           [a['label'] for a in r['actions']], adapter=adapter)
        pred = max(range(len(p)), key=p.__getitem__)
        rows.append(dict(state=r['state_hash'], task=r['task'], kind=r['kind'], n_options=len(p), correct=pred == r['gold'],
                         pred_kind=r['actions'][pred]['kind'], p_gold=p[r['gold']], equivalent=equivalent(r, pred)))
    return pd.DataFrame(rows)


def evaluate_values(records, adapter, desc):
    rows = []
    for r in tqdm(records, desc=desc, mininterval=10):
        p = with_oom_retry(score_options, model, TOKENIZER, LETTERS, value_context(r['goal'], r['url'], r['field'], r['history']),
                           VALUE_QUESTION, r['values'], adapter=adapter)
        rows.append(dict(state=r['state_hash'], task=r['task'], correct=max(range(len(p)), key=p.__getitem__) == r['value_gold'],
                         n_options=len(p)))
    return pd.DataFrame(rows)


def cached_frame(name, fn):
    path = RUN_DIR / f'{name}.csv'
    if path.exists():
        return pd.read_csv(path)
    df = fn()
    df.to_csv(path, index=False)
    return df

In [ ]:
# @title Held-out: Basis vs. veröffentlichter v2 vs. eigener Adapter
t0 = time.time()
CONDITIONS = {'Basis': False, 'v2 (veröffentlicht)': 'v2', 'eigenes': True}
KEYS = dict(zip(CONDITIONS, ['base', 'v2', 'own']))
ev = {name: cached_frame(f'heldout_{KEYS[name]}', lambda a=a, n=name: evaluate_actions(eval_records, a, n))
      for name, a in CONDITIONS.items()}
evv = {name: cached_frame(f'heldout_values_{KEYS[name]}', lambda a=a, n=name: evaluate_values(eval_values, a, n + ', Texte'))
       for name, a in CONDITIONS.items()}
TIMINGS['heldout_eval_s'] = time.time() - t0


def accuracy_table(frames, value_frames):
    first = next(iter(frames.values()))
    rows = []
    for k in KINDS + ['gesamt']:
        sel = (lambda d: d) if k == 'gesamt' else (lambda d, k=k: d[d.kind == k])
        row = {'Aktion': k, 'n': len(sel(first)), 'Raten': (1 / sel(first).n_options).mean()}
        for name, d in frames.items():
            row[name] = sel(d).correct.mean()
        rows.append(row)
    vf = next(iter(value_frames.values()))
    row = {'Aktion': 'Texteingabe: Wert', 'n': len(vf), 'Raten': (1 / vf.n_options).mean()}
    row.update({name: d.correct.mean() for name, d in value_frames.items()})
    rows.append(row)
    return pd.DataFrame(rows).set_index('Aktion')


heldout_table = accuracy_table(ev, evv)
display(heldout_table.style.format({c: '{:.1%}' for c in heldout_table.columns if c != 'n'}))
kept = {name: d[d.task.map(task_kept)] for name, d in ev.items()}
extra = pd.DataFrame({
    f'nur vom Filter behaltene Aufgaben (n={len(kept["eigenes"])})': {n: d.correct.mean() for n, d in kept.items()},
    'gleichwertiges Ziel (sekundär)': {n: d.equivalent.mean() for n, d in ev.items()},
}).T
display(extra.style.format('{:.1%}'))
print('Vorhergesagter Aktionstyp je demonstriertem Typ (eigener Adapter):')
display(pd.crosstab(ev['eigenes'].kind, ev['eigenes'].pred_kind, margins=True))


def sign_test(a, b):
    only_a, only_b = int((a & ~b).sum()), int((~a & b).sum())
    p = math.erfc(abs(only_a - only_b) / math.sqrt(max(1, only_a + only_b)) / math.sqrt(2))
    return only_a, only_b, p


for x, y in [('eigenes', 'Basis'), ('eigenes', 'v2 (veröffentlicht)')]:
    a, b, p = sign_test(ev[x].correct.values, ev[y].correct.values)
    print(f'Nur {x} richtig: {a} | nur {y} richtig: {b} | Vorzeichentest p ≈ {p:.2g}')

### Präzisionsprobe: Stimmen die Entscheidungen mit dem BF16-Referenzlauf überein?
Der volle Lauf hat Basis und v2 auf denselben 600 Held-out-Entscheidungen in BF16 ausgewertet (RTX A6000). Hier stehen dessen
Ergebnisse je Entscheidung (richtig/falsch und Wahrscheinlichkeit der demonstrierten Aktion, auf 16 Bit quantisiert). Laufen
Basis und v2 in dieser Laufzeit – etwa in FP16 auf einer T4 – auf dieselben Entscheidungen hinaus, ist die Präzision kein Problem.

In [ ]:
# @title Vergleich mit dem BF16-Referenzlauf (je Entscheidung)
REFERENCE_BF16 = {"base": {"correct": "1100800000000000020000001000000000308000020002590420049000000000c0048080480000840020001020820010040010000000214040000000400000040000010210000220404024", "p_gold": "MAS+ALVKjZDjAF0AKwBWDgQArhOPAAIAZgGpBhsANQECv5oAAgAFAEUBagABACYDcwUTEb8ENQE6AtYASwE0ACoFAQBFAIEBIQMtAcwCkwAtAFsQ8QQBAHIBWRwUADAA3QBrBA8fVgB7ADQD0BhYAJ0KBgAsAiMAHgUBADgCMgQiAPgyYwxvBZYTJRyWv00IogEWBFwBAALOIgACbAZZD10DF1ENAHwA6wJGAQIA/gEXAAEA6QHpBg4AYAH2AJ4E0xcbAHAImPZ0AJcAEgk9ABEDxxUQAQMAOwD4Ad8KsAIWBkYBQQKbALknxgIoB8gBjQUqAf4AegGEAFwFBQAeALEFbgdjAz0dwgAAAOs1BhB4ERgAFBH/hJoB5AM4AOELLuAMAzUCJgAxA7MBTQE6AOQXNAOGAPMAIQGNAEAADBFwBrshZggLALIjSgTrvx4APQA/AZcKHQB+BAYLLwGYAPwBDACqFJYACwA8AtcoOQH/AFH98gC0XjD7WQBAAAveFAHnBSMFgAAFABpEEAC6ADIJrQtrQB0AWgCnAQcAhQlgHjoHPQALALABFEEuAhcAFz4ZAGEK+MQ/C9cCAQAxAEQBaAapAKsAmQsoDAEAJQDbCAQAFgGOAOgA1gBAAAYAcQCgAJwAjwEXALYBJQMZBxwDOwQjDWwMAAC0AO0A0QJsOcxmVgAaAbEAFAAjA44CTQACALcAkwBvAls2ehcqA8Tl1gYSAAoAaADuDdUADQB+3xwEHAC4DyQXqxA0AaAALRNlHTwAGwH8xgABowDDAQkBFADTAJkAOAHRABgAEQASADwA+QUyEzEFBge+A5YB/dsAAC4A9wUWAHu0HgDFB6IBUwARAwwCNkPbJ00QhwI0CFgEZejPAo4BlQDbBYgAoQCdAB4APg7IAVsAdQhjANcATwETBsmKcwA+AFMRNwFKAtwAiOnsHU0AGgDbAHEJTTdlABMSaQSsBE8CTvhEAicAggAbAJgBJwUfCkYADwABAesA9gD3xC0ABQA7AOQB1gIAANAAmQUCABznSgAPAGYDfgMAAHIXjA/lGpoAwgNTAScJhgfdxzMArgD1JT4BCwAJFQQASwrBBLwH6wACAKwAaQAwAMIDxQxHAHIcUgh1AXwEoAeCBikAAQAVBLQKKQAIAws7CAApAFIDpAk6MjQBzHsmAI0CRRIvALEALwEAAL2RhAAADAAAbAjaEAYDZQA1AM8B/hIEDAgB0QBpARkZgRUiAMICBgCkAEoAAgAcAogCKwGiKQEADwA2AO0VlgTNmrUQEwD7AEoAHQEqANkAwAQNAFQagwJmAD0AhwUBAHwBsg3SAiMAzgIGACkAPgCLAUsAFwAZFhrwVQROAH0D8QDtA10AAgAfDbYJegDmEYoAFw0XANkGtgHtFgEDdwBMASsA4wAPCVIAWAIEzBQAYwaEAJ4CCwMEANFqGgANAeoAOwC5fysBAwD1AWAACAA3AfcC2QjmCbgAFACHHbsBIwJ2AEkQeQIhAC5W1AkOAT0AUCUCADoDkQEIBgAAvQCQHTwAKgBpHDQFDAB/FgEOwP4VAAUQLQA/AKcHnQALAJMAonEUAT0skSZBHPMN", "values": "80042822001a40058b0008b2012422a0a841440050469000801862003b20883010200400a0870140181835a00204a3021168"}, "v2": {"correct": "63e10ce5a8c76084374792795ba4c4bf101891e62ec05378aba850c41df94417c02c89aa5158b7c087e068f5b3111e4538001b152224330c8f4880140f1ced04e4a20453ece9d4020dec99", "p_gold": "ZmLeRGpzbliyDmMMJ3AcPUza6H66S90Q+woeBGQcXqjACfsT/QFNEeNfgtAIABE0CjJfr05iCA34A3lplSG+6t1sCgCzakwDrT0jIucpQRsJZNmYAhOjAe4T1N3wmZJ0hACuNB3c7wHoAEI43gLBTMhqIgGBBjwSuQHLMX8T7AMRAPsOaX/hw9oBd+pRmT480ghuniEAxQ3FRkZAoFG4155PBSMeDjHkcQI4K4WzMAN/BW1RrXmNmhlMNE98JGVP7BVR7KwhU4u/ug4JgqecbiWToQU8X9NajRdxi+QZrgUybN1KoAC/AsgEV8MqCuwEZXQtALN8q53Y2JlTVTZ/zSglNATIEcDRCwA6AWAD+QUyAcsB1gCtq47Itk2mAN0E664JOqYYDWwLI+gGCw7tVK6FFekDqqMdlwF+SNp/JwRWAdkAwtttAfC94KOcs8Jp9io8k8M34AxDGrwXygCYAU0t2LY7CPWSnzW0BGZX7UHtBKTsZVDk1Qq8wUvUHag2a/apIpqaWgkpn80ALZnEPSVdTALopzgAtTpeCxYAXgwNIUH3kgCcKMgLxgKIHXUAB3xrL0IB9wM1BapzFQDFA9IKpACQA77NvS5v6B8AUq34QlS3lcHkeb1ziAVjAO2uEAAOkL8qtwRKBGhzlgMgV/cXbwYqDHrwiwMTeKGitnqH2bI3nBUyALADmBe7CRABWCY0AJx9MAHjf2OcUwFMBAY31Tv2BwwBzt56NUoUQcahtH8e4EHgAq3m1gJBnuEbmhtuqUIdHeUUQwwCRBtMU5AEKt3/AP6ARjuhAMMWrACsYJ8HJLk1ahoEGNgvraGflJbOz7QSWwFGEDcQEwBICfbXJQjiBZETtgJ2UxJVDqzOZNx8KqFcAAsBkwAlI5YCSALFz5aRBRXTMYcNBQCzB3P5Mb6t18lmggQMgDsBb5+/LzUAvXYicpYImATEVu/pGQC9KD8NeujqKIoaxj19vs4WngsjFbLOCVe586IjuRdOAnyWuBaxEYhIVVMjAODvIAAzADI+WqHKs38URAAnHxMZIgCMAT8MuQJnAnEj0gYIBpAuXQCXXGB0ngdmjINZsgnfFt8vYfIeBF9oSiwBRsQAzgzIoX8COAQOE+33dwycQqMFTezRAW4ATExjAZsIIjt0JH6BZp2pBJINY/JW4eEj6jqqBXcW3TqelCQAAxZSliMgQRfhAR7C6i8S7pXl4AjByb4AgxgV6DwdvgeeAflppj4sKlEXmQ0aADQB5jJKMSsWGQCSzhUAyNRfJ+4dggolPdwkEgDVWs3tzON2a7FefxHCATX535pi3jwKRgZ9xbDBsUkONRc0zPSJKzHc+wBYAAcO51meNraiCQPeAsYkhLxyRlclIABw9JoGfUErwlwyf870ChASMAwUW5AnngSGBngu2wzlAslBaBIfLJkAkqUfBW7Guh4wAMPDVh46TlyzoCbsTWA6+eNOApIW0ImclcptSgTwQokILxtuXg2CKJWIAOjy+QKl7D4eGAEKD9UBNyBEBqMhZQmy+WUZ/BAWIycM2wGH+mMkDgAf6z7vs9Ke4FQOJTLBUmsCawEa9U8AUxGTscyujwhyBcBc", "values": "7edee5fffefee6fd9fa7df5bdd33bffd7ff95eba7ef7affdefb8f6eebf3fbf7efb7fdb35fbfd1defbd7efbecf27ffb5c9d7d"}}


def ref_bits(hexstr, n):
    return [c == '1' for c in bin(int(hexstr, 16))[2:].zfill(n)]


rows = []
for name, key in [('Basis', 'base'), ('v2 (veröffentlicht)', 'v2')]:
    ref = REFERENCE_BF16[key]
    ok = ref_bits(ref['correct'], N_EVAL_ALL)[:len(ev[name])]
    p_ref = [v / 65535 for v in struct.unpack(f'<{N_EVAL_ALL}H', base64.b64decode(ref['p_gold']))][:len(ev[name])]
    vok = ref_bits(ref['values'], N_VALUE_EVAL)
    here = ev[name]
    dp = (here.p_gold - pd.Series(p_ref)).abs()
    rows.append({'Bedingung': name, 'Präzision hier': PRECISION, 'n': len(here),
                 'Genauigkeit hier': here.correct.mean(), 'Genauigkeit BF16-Referenz': sum(ok) / len(ok),
                 'gleiches Ergebnis je Entscheidung': (here.correct.values == pd.Series(ok).values).mean(),
                 '|Δp(demonstriert)| Median': dp.median(), '|Δp| Max': dp.max(),
                 'Texte hier / Referenz': f'{evv[name].correct.mean():.1%} / {sum(vok) / len(vok):.1%}',
                 'NaN': int(here.p_gold.isna().sum())})
precision_check = pd.DataFrame(rows).set_index('Bedingung')
precision_check.to_csv(RUN_DIR / 'precision_check.csv')
display(precision_check.style.format({'Genauigkeit hier': '{:.1%}', 'Genauigkeit BF16-Referenz': '{:.1%}',
                                      'gleiches Ergebnis je Entscheidung': '{:.1%}', '|Δp(demonstriert)| Median': '{:.4f}',
                                      '|Δp| Max': '{:.3f}'}))

## 10 · Live: ein Crawler auf echten Websites
Aufgaben auf öffentlichen Übungsseiten für Scraper (`books.toscrape.com`, `quotes.toscrape.com`, `webscraper.io/test-sites`,
`scrapethissite.com`). Jede Aufgabe hat ein prüfbares Ziel (Zielseite bzw. Seiteninhalt) und eine Datenfrage. Der Crawler:

1. liest den BrowserGym-Baum, erzeugt die Kandidaten, entscheidet mit **einem** Forward Pass,
2. wählt bei `type` den Text in einer zweiten, kurzen Entscheidung,
3. führt die Aktion in Chromium aus und hängt sie im NNetNav-Format an die Historie,
4. extrahiert nach `stop` den gesuchten Wert: wieder eine Auswahl – unter den Texten der Seite.

Bedingungen: veröffentlichter v2 und der eigene Adapter, das Basismodell optional (`LIVE_BASE`). Die Extraktion läuft überall mit
dem Basismodell (Adapter aus), damit sich die Bedingungen nur in der Navigation unterscheiden. **Erfolg** heißt: gestoppt auf der
Zielseite. **Daten korrekt** heißt zusätzlich: extrahierter Wert stimmt. Referenz des vollen Laufs: Basis 6/16, v2 13/16
(Daten 12/16).

Für die Korrekturschleife gibt es zu jeder Aufgabe einen aufgezeichneten Referenzablauf (Link-Ziele bzw. Feld und Suchtext) – so,
wie ein Playwright-Recorder ihn für einen bestehenden Crawler liefert. Er wird nur für die separaten Korrekturaufgaben benutzt,
nie für die Prüfaufgaben.

In [ ]:
# @title Browser-Laufzeit: Locator, Ausführung, Seitentexte
URL_ATTR_RE = re.compile(r"url='([^']*)'")


def node_urls(tree):
    out = {}
    for line in tree.split('\n'):
        m = NODE_RE.match(line)
        if m:
            u = URL_ATTR_RE.search(m[6])
            if u and m[2] not in out:
                out[m[2]] = u[1]
    return out


def locate(page, bid):
    """BrowserGym-bid -> Playwright-Locator, auch in iFrames (bid-Präfix aus Kleinbuchstaben = Frame)."""
    frame, i = page, 0
    while bid[i:] and not bid[i:].isnumeric():
        i += 1
        while bid[i:] and bid[i].isalpha() and bid[i].isupper():
            i += 1
        frame = frame.frame_locator(f'[bid="{bid[:i]}"]')
    return frame.locator(f'[bid="{bid}"]')


async def settle(page):
    for state, timeout in (('load', 15000), ('networkidle', 5000)):
        try:
            await page.wait_for_load_state(state, timeout=timeout)
        except Exception:
            pass
    await page.wait_for_timeout(500)


async def observe_page(page, js, tries=5):
    last = None
    for _ in range(tries):
        try:
            await page.wait_for_load_state('domcontentloaded', timeout=15000)
            tree, _, _ = await bgym_observe(page, js)
            return tree
        except Exception as exc:      # Navigation während der Markierung: erneut lesen (wie BrowserGym)
            last = exc
            await page.wait_for_timeout(700)
    raise last


async def execute(page, action, text=''):
    k = action['kind']
    if k == 'click':
        await locate(page, action['bid']).click(timeout=8000)
    elif k == 'type':
        target = locate(page, action['bid'])
        await target.fill(text, timeout=8000)
        if action['enter']:
            await target.press('Enter', timeout=8000)
    elif k == 'scroll':
        await page.mouse.wheel(0, 600 if action['direction'] == 'down' else -600)
    elif k == 'go_back':
        await page.go_back(wait_until='domcontentloaded', timeout=20000)
    await settle(page)


def page_texts(tree, limit=MAX_OPTIONS):
    """Textbausteine einer Seite als Extraktionsoptionen (Dokumentreihenfolge, dedupliziert)."""
    seen, out = set(), []
    for line in tree.split('\n'):
        m = re.match(r"^\t*(?:\[[A-Za-z0-9]+\] )?(StaticText|cell|gridcell|heading|paragraph|link|listitem|term|definition) "
                     r"(?:'((?:[^'\\]|\\.)*)'|\"((?:[^\"\\]|\\.)*)\")", line)
        if not m:
            continue
        raw, q = (m[2], "'") if m[2] is not None else (m[3], '"')
        try:
            text = ast.literal_eval(q + raw + q).strip()
        except (ValueError, SyntaxError):
            text = raw.strip()
        if len(re.sub(r'[\W_]+', '', text)) < 1 or text in seen:
            continue
        seen.add(text)
        out.append(short(text, 200))
    return out[:limit]

In [ ]:
# @title Aufgaben, Referenzabläufe, Crawl-Schleife
def T(tid, start, goal, path, done_url=None, done_text=None, question=None, answer=None):
    return dict(id=tid, start=start, goal=goal, path=path, done_url=done_url, done_text=done_text,
                question=question, answer=answer)

BOOKS, QUOTES = 'https://books.toscrape.com/', 'https://quotes.toscrape.com/'
WS = 'https://webscraper.io/test-sites/e-commerce/allinone'
STS = 'https://www.scrapethissite.com/pages/'


def task_done(task, url, tree):
    ok = True
    if task['done_url']:
        ok &= re.search(task['done_url'], url) is not None
    if task['done_text']:
        ok &= task['done_text'] in tree
    return ok


def reference_action(task, url, tree, actions, nodes):
    """Aufgezeichneter Referenzablauf -> (bevorzugte Aktion, Text, alle Aktionen, die dem Ablauf folgen).
    Bevorzugt wird der weiteste noch erreichbare Schritt; ohne passenden Schritt: zurück."""
    stop = next(i for i, a in enumerate(actions) if a['kind'] == 'stop')
    if task_done(task, url, tree):
        return stop, '', {stop}
    urls, ok, best = node_urls(tree), set(), None
    for hop_index, (kind, spec) in enumerate(task['path']):
        for i, a in enumerate(actions):
            n = nodes.get(a.get('bid'), {})
            hit = False
            if kind == 'click' and a['kind'] == 'click':
                hit = ('url' in spec and re.search(spec['url'], urls.get(a['bid'], ''))) or \
                      ('name' in spec and n.get('role') in ('link', 'button') and re.fullmatch(spec['name'], n.get('name', '')))
            elif kind == 'type' and a['kind'] == 'type' and a['enter']:
                hit = re.search(spec['field'], n.get('name', '')) is not None
            if hit:
                ok.add(i)
                if best is None or hop_index >= best[0]:
                    best = (hop_index, i, spec.get('value', ''))
    if best is None:
        back = next(i for i, a in enumerate(actions) if a['kind'] == 'go_back')
        return back, '', {back}
    return best[1], best[2], ok


async def run_episode(browser, js, task, choose_action, choose_value, max_steps=12, reviewer=None, verbose=False):
    """Ein Crawl: Beobachten -> Entscheiden -> Ausführen, bis stop oder Schrittbudget.
    reviewer: Funktion für die Korrekturrunde; dann wird die Referenzaktion ausgeführt und das Paar gespeichert."""
    context = await browser.new_context(viewport={'width': 1280, 'height': 720})
    page = await context.new_page()
    history, steps, status = ['None'], [], 'step_budget'
    t0 = time.perf_counter()
    try:
        await page.goto(task['start'], wait_until='domcontentloaded', timeout=45000)
        await settle(page)
        for step in range(max_steps):
            if len(context.pages) > 1:            # neuer Tab geöffnet -> dort weiter (wie BrowserGym)
                page = context.pages[-1]
                await settle(page)
            tree = await observe_page(page, js)
            url = page.url
            actions, nodes = candidates(tree)
            if len(actions) > MAX_OPTIONS:
                status = 'too_many_actions'
                break
            user_text = observation_text(tree, url, task['goal'], history)
            t_dec = time.perf_counter()
            probs = choose_action(user_text, actions)
            pick = max(range(len(actions)), key=probs.__getitem__)
            value, values = '', []
            numbered = [f'{i + 1}: {h}' for i, h in enumerate(history)]
            if actions[pick]['kind'] == 'type':
                values = value_candidates(task['goal'], numbered)
                n = nodes[actions[pick]['bid']]
                field = f"[{actions[pick]['bid']}] {n['role']} {short(n['name'])!r}"
                vp = choose_value(value_context(task['goal'], url, field, numbered), values)
                value = values[max(range(len(values)), key=vp.__getitem__)]
            decide_ms = 1000 * (time.perf_counter() - t_dec)
            rec = dict(step=step, url=url, goal=task['goal'], user_text=user_text, history=numbered, actions=actions,
                       pick=pick, value=value, p=probs[pick], decide_ms=decide_ms, tree_lines=tree.count('\n') + 1)
            chosen, text = pick, value
            if reviewer is not None:
                ref, ref_text, acceptable = reviewer(task, url, tree, actions, nodes)
                if pick in acceptable and (actions[pick]['kind'] != 'type' or norm_text(value) == norm_text(ref_text)):
                    ref, ref_text = pick, value                  # Modell folgt dem Ablauf: keine Korrektur
                elif actions[pick]['kind'] == 'type' and pick in acceptable:
                    ref = pick                                   # richtiges Feld, falscher Text: nur der Text wird korrigiert
                rec.update(reference=ref, reference_value=ref_text, corrected=(ref, ref_text) != (pick, value),
                           values=value_candidates(task['goal'], numbered))
                if actions[ref]['kind'] == 'type':
                    n = nodes[actions[ref]['bid']]
                    rec['field'] = f"[{actions[ref]['bid']}] {n['role']} {short(n['name'])!r}"
                chosen, text = ref, ref_text
            steps.append(rec)
            if verbose:
                print(f"  {step + 1:>2} {decide_ms:6.0f} ms  {actions[pick]['label'][:90]}" + (f' <- "{value}"' if value else '')
                      + (f"   | Referenz: {actions[chosen]['label'][:60]}" if reviewer is not None and chosen != pick else ''))
            action = actions[chosen]
            if action['kind'] == 'stop':
                status = 'stopped'
                break
            try:
                await execute(page, action, text)
            except Exception as exc:
                rec['error'] = f'{type(exc).__name__}: {str(exc)[:200]}'
                status = 'execution_error'
                break
            history.append(history_entry(action, nodes, text))
        final_tree = await observe_page(page, js)
        final_url = page.url
    finally:
        await context.close()
    reached = task_done(task, final_url, final_tree)
    return dict(task=task['id'], goal=task['goal'], status=status, steps=steps, final_url=final_url,
                final_tree=final_tree, success=status == 'stopped' and reached, reached_target=reached,
                seconds=time.perf_counter() - t0)

In [ ]:
# @title Aufgaben
EVAL_TASKS = [   # Prüfaufgaben: nie für Korrekturen verwendet
    T('books-sharp-objects', BOOKS, "Find the price of the book 'Sharp Objects' on Books to Scrape.",
      [('click', {'url': 'sharp-objects_997/'})], done_url=r'sharp-objects_997/index\.html',
      question='What is the price of the book?', answer='£47.82'),
    T('books-soumission', BOOKS, "Find the price of the book 'Soumission'.",
      [('click', {'url': 'soumission_998/'})], done_url=r'soumission_998/index\.html',
      question='What is the price of the book?', answer='£50.10'),
    T('books-requiem-stock', BOOKS, "How many copies of 'The Requiem Red' are available?",
      [('click', {'url': 'the-requiem-red_995/'})], done_url=r'the-requiem-red_995/index\.html',
      question='How many copies are available?', answer='19 available'),
    T('books-travel-vagabonding', BOOKS, "Open the Travel category and find the price of 'Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel'.",
      [('click', {'url': 'travel_2/'}), ('click', {'url': 'vagabonding-an-uncommon-guide'})],
      done_url=r'vagabonding-an-uncommon-guide.*index\.html', question='What is the price of the book?', answer='£36.94'),
    T('books-mystery-dark-wood', BOOKS, "Find the price of the mystery novel 'In a Dark, Dark Wood' in the Mystery category.",
      [('click', {'url': 'mystery_3/'}), ('click', {'url': 'in-a-dark-dark-wood'})],
      done_url=r'in-a-dark-dark-wood.*index\.html', question='What is the price of the book?', answer='£19.63'),
    T('quotes-einstein-born', QUOTES, 'Find the birth date of Albert Einstein.',
      [('click', {'url': 'author/Albert-Einstein'})], done_url=r'/author/Albert-Einstein',
      question='When was the author born?', answer='March 14, 1879'),
    T('quotes-austen-place', QUOTES, 'Find out where Jane Austen was born.',
      [('click', {'url': 'author/Jane-Austen'})], done_url=r'/author/Jane-Austen',
      question='Where was the author born?', answer='Steventon Rectory'),
    T('quotes-tag-humor', QUOTES, "Show the quotes tagged 'humor'.",
      [('click', {'url': '/tag/humor/'})], done_url=r'/tag/humor/',
      question='Who wrote the first quote on the page?', answer='Jane Austen'),
    T('quotes-page-2', QUOTES, 'Go to the second page of quotes and find the author of the first quote there.',
      [('click', {'url': '/page/2/'})], done_url=r'/page/2/',
      question='Who wrote the first quote on the page?', answer='Marilyn Monroe'),
    T('ws-lenovo-ideatab', WS, "Find the price of the tablet 'Lenovo IdeaTab' in the web scraper test shop.",
      [('click', {'url': 'allinone/computers'}), ('click', {'url': 'computers/tablets'}), ('click', {'url': r'product/10$'})],
      done_url=r'/product/10$', question='What is the price of the product?', answer='$69.99'),
    T('ws-nokia-123', WS, "How many reviews does the phone 'Nokia 123' have?",
      [('click', {'url': 'allinone/phones'}), ('click', {'url': 'phones/touch'}), ('click', {'url': r'product/1$'})],
      done_url=r'/product/1$', question='How many reviews does the product have?', answer='11'),
    T('sts-open-hockey', STS, 'Open the hockey teams page on Scrape This Site.',
      [('click', {'url': '/pages/forms/'})], done_url=r'/pages/forms/',
      question='How many items are listed?', answer='25 items'),
    T('sts-hockey-detroit', STS + 'forms/', "Search for the hockey team 'Detroit Red Wings' and find how many games they won in 1990.",
      [('type', {'field': 'Search', 'value': 'Detroit Red Wings'})], done_url=r'q=Detroit',
      question='How many wins did the team have in 1990?', answer='34'),
    T('sts-oscars-2012', STS + 'ajax-javascript/', 'Show the Oscar winning films of 2012.',
      [('click', {'name': r'2012'})], done_text="'Argo'", question='Which film won Best Picture?', answer='Argo'),
    T('sts-countries-andorra', STS + 'simple/', 'Find the capital of Andorra.',
      [], done_url=r'/pages/simple/', question='What is the capital of Andorra?', answer='Andorra la Vella'),
    T('sts-open-turtles', STS, "Open the page 'Turtles All the Way Down: Frames & iFrames'.",
      [('click', {'url': '/pages/frames/'})], done_url=r'/pages/frames/',
      question='How many turtle families are shown?', answer='14 items'),
]
CORRECTION_TASKS = [   # Korrekturaufgaben: andere Ziele auf denselben Sites
    T('c-books-tipping', BOOKS, "Find the price of the book 'Tipping the Velvet'.",
      [('click', {'url': 'tipping-the-velvet_999/'})], done_url=r'tipping-the-velvet_999/index\.html',
      question='What is the price of the book?', answer='£53.74'),
    T('c-books-poetry', BOOKS, "Open the Poetry category and find the price of 'Shakespeare's Sonnets'.",
      [('click', {'url': 'poetry_23/'}), ('click', {'url': 'shakespeares-sonnets'})],
      done_url=r'shakespeares-sonnets.*index\.html', question='What is the price of the book?', answer='£20.66'),
    T('c-books-attic-stock', BOOKS, "How many copies of 'A Light in the Attic' are available?",
      [('click', {'url': 'a-light-in-the-attic_1000/'})], done_url=r'a-light-in-the-attic_1000/index\.html',
      question='How many copies are available?', answer='22 available'),
    T('c-quotes-rowling', QUOTES, 'Find the birth date of J.K. Rowling.',
      [('click', {'url': 'author/J-K-Rowling'})], done_url=r'/author/J-K-Rowling',
      question='When was the author born?', answer='July 31, 1965'),
    T('c-quotes-tag-love', QUOTES, "Show the quotes tagged 'love'.",
      [('click', {'url': '/tag/love/'})], done_url=r'/tag/love/',
      question='Who wrote the first quote on the page?', answer='André Gide'),
    T('c-ws-sony-xperia', WS, "Find the price of the phone 'Sony Xperia' in the web scraper test shop.",
      [('click', {'url': 'allinone/phones'}), ('click', {'url': 'phones/touch'}), ('click', {'url': r'product/5$'})],
      done_url=r'/product/5$', question='What is the price of the product?', answer='$118.99'),
    T('c-sts-hockey-boston', STS + 'forms/', "Search for the hockey team 'Boston Bruins'.",
      [('type', {'field': 'Search', 'value': 'Boston Bruins'})], done_url=r'q=Boston',
      question='How many wins did the team have in 1990?', answer='44'),
    T('c-sts-oscars-2014', STS + 'ajax-javascript/', 'Show the Oscar winning films of 2014.',
      [('click', {'name': r'2014'})], done_text="'Birdman'", question='Which film won Best Picture?', answer='Birdman'),
    T('c-sts-countries-afghanistan', STS + 'simple/', 'Find the capital of Afghanistan.',
      [], done_url=r'/pages/simple/', question='What is the capital of Afghanistan?', answer='Kabul'),
    T('c-sts-open-countries', STS, 'Open the countries of the world page on Scrape This Site.',
      [('click', {'url': '/pages/simple/'})], done_url=r'/pages/simple/',
      question='How many countries are listed?', answer='250 items'),
]

In [ ]:
# @title Entscheider, Extraktion, Aufgabenserien
def deciders(adapter):
    act = lambda ctx, acts: with_oom_retry(score_options, model, TOKENIZER, LETTERS, ctx, ACTION_QUESTION,
                                           [a['label'] for a in acts], adapter=adapter)
    val = lambda ctx, values: with_oom_retry(score_options, model, TOKENIZER, LETTERS, ctx, VALUE_QUESTION,
                                             values, adapter=adapter)
    return act, val


def extract(task, result):
    """Datenextraktion als Auswahl unter den Texten der Endseite (Basismodell, Adapter aus)."""
    if result['status'] != 'stopped':
        return None
    texts = page_texts(result['final_tree'])
    ctx = f"OBJECTIVE: {task['goal']}\nURL: {result['final_url']}\nPAGE:\n{result['final_tree']}"
    p = with_oom_retry(score_options, model, TOKENIZER, LETTERS, ctx, EXTRACT_QUESTION.format(q=task['question']),
                       texts, adapter=False)
    return texts[max(range(len(p)), key=p.__getitem__)]


def answer_ok(expected, got):
    if got is None:
        return False
    e, g = norm_text(expected), norm_text(got)
    return e == g or (len(e) > 4 and e in g)


async def run_suite(tasks, adapter, name, reviewer=None, verbose_first=False):
    act, val = deciders(adapter)
    results = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        for i, task in enumerate(tasks):
            for attempt in range(3):                  # Netzwerkfehler der Website, nicht Fehler des Modells
                try:
                    r = await run_episode(browser, BGYM_JS, task, act, val, CFG['max_steps'], reviewer,
                                          verbose=verbose_first and i == 0)
                    break
                except Exception as exc:
                    if attempt == 2:
                        raise
                    print('  Wiederholung wegen', type(exc).__name__, str(exc)[:120])
                    await asyncio.sleep(10)
            r['condition'] = name
            r['extracted'] = extract(task, r) if reviewer is None else None
            r['data_ok'] = r['success'] and answer_ok(task['answer'], r['extracted'])
            results.append(r)
            print(f"{name:>16} | {task['id']:<28} {r['status']:<15} {'Ziel' if r['success'] else '–':<5}"
                  f"{len(r['steps']):>3} Schritte {r['seconds']:6.1f}s -> {r['extracted']!r}"[:170])
        await browser.close()
    return results


async def suite(key, label, tasks, adapter, keep_text=False, **kw):
    """Ergebnisse werden gespeichert; ein erneuter Notebook-Lauf lädt sie statt neu zu crawlen."""
    path = RUN_DIR / f'live_{key}.json'
    if path.exists():
        return json.loads(path.read_text())
    res = await run_suite(tasks, adapter, label, **kw)
    for r in res:
        r.pop('final_tree' if not keep_text else '_', None)
        for s in r['steps']:
            if not keep_text:
                s.pop('user_text', None)
    path.write_text(json.dumps(res, ensure_ascii=False))
    return res


def suite_frame(results, tasks):
    by = {t['id']: t for t in tasks}
    return pd.DataFrame([dict(Aufgabe=r['task'], Bedingung=r['condition'], Status=r['status'], Erfolg=r['success'],
                              Schritte=len(r['steps']), Frage=by[r['task']]['question'], Extrahiert=r['extracted'],
                              Erwartet=by[r['task']]['answer'], Daten_korrekt=r['data_ok'], URL=r['final_url'],
                              ms_je_Entscheidung=sum(s['decide_ms'] for s in r['steps']) / max(1, len(r['steps'])))
                         for r in results])


print(len(EVAL_TASKS), 'Prüfaufgaben,', len(CORRECTION_TASKS), 'Korrekturaufgaben')

In [ ]:
# @title Prüfaufgaben: veröffentlichter v2 vs. eigener Adapter (Basis optional)
t0 = time.time()
live = {}
if CFG['live_base']:
    live['Basis'] = await suite('eval_base', 'Basis', EVAL_TASKS, False)
live['v2 (veröffentlicht)'] = await suite('eval_v2', 'v2 (veröffentlicht)', EVAL_TASKS, 'v2')
live['eigenes'] = await suite('eval_own', 'eigenes', EVAL_TASKS, True, verbose_first=True)
TIMINGS['live_eval_s'] = time.time() - t0
live_df = pd.concat([suite_frame(v, EVAL_TASKS) for v in live.values()])
display(live_df.pivot_table(index='Aufgabe', columns='Bedingung', values=['Erfolg', 'Daten_korrekt', 'Schritte'],
                            aggfunc='first'))
display(live_df.groupby('Bedingung')[['Erfolg', 'Daten_korrekt', 'Schritte', 'ms_je_Entscheidung']].agg(['sum', 'mean']).round(3))

In [ ]:
# @title Crawl-Ergebnis: die gesammelten Daten (eigener Adapter)
crawl_data = live_df[live_df.Bedingung == 'eigenes'][['Aufgabe', 'URL', 'Frage', 'Extrahiert', 'Erwartet', 'Daten_korrekt']]
crawl_data.to_json(RUN_DIR / 'crawl_data_own.jsonl', orient='records', lines=True, force_ascii=False)
display(crawl_data)

## 11 · Korrigieren und weitertrainieren (optional, `RUN_CORRECTIONS`)
Auf den **Korrekturaufgaben** (andere Bücher, Autoren, Produkte, Suchbegriffe auf denselben Sites) läuft der Crawler mit dem
eigenen Adapter. An jedem Zustand wird seine Wahl mit dem aufgezeichneten Referenzablauf verglichen; ausgeführt wird die
Referenzaktion, damit der Lauf auf Kurs bleibt und weitere Zustände liefert (DAgger). Abweichungen sind Korrekturen. Das Label
kommt aus dem Referenzablauf, nicht aus dem Modell. Die Tabelle ist zur Durchsicht gedacht: Einträge in `REJECT` werden verworfen.
Weitertrainiert wird mit den geprüften Zuständen im Tokenbudget des Profils plus dreimal so viel Replay aus den eigenen
Trainingsentscheidungen, 3 Epochen.

In [ ]:
# @title Korrekturen sammeln und durchsehen
REJECT = set()   # z. B. {('c-books-poetry', 1)}
if not RUN_CORRECTIONS:
    print('RUN_CORRECTIONS ist aus: Abschnitt 11 wird übersprungen.')
else:
    corr_before = await suite('corr_own', 'eigenes', CORRECTION_TASKS, True)
    reviewed = await suite('corr_review', 'Durchsicht', CORRECTION_TASKS, True, keep_text=True, reviewer=reference_action)
    review_rows, reviewed_records = [], []
    for r in reviewed:
        for s in r['steps']:
            acts, ref = s['actions'], s['reference']
            corrected = s['corrected']
            review_rows.append(dict(Aufgabe=r['task'], Schritt=s['step'], URL=s['url'],
                                    Modell=acts[s['pick']]['label'][:60] + (f' "{s["value"]}"' if s['value'] else ''),
                                    Referenz=acts[ref]['label'][:60] + (f' "{s["reference_value"]}"' if s['reference_value'] else ''),
                                    Korrektur=corrected))
            if (r['task'], s['step']) in REJECT:
                continue
            rec = dict(task=r['task'], goal=s['goal'], url=s['url'], history=s['history'], user_text=s['user_text'],
                       actions=acts, gold=ref, kind=acts[ref]['kind'], corrected=corrected, label_source='recorded_reference_flow')
            if rec['kind'] == 'type':
                rec.update(field=s['field'], values=s['values'],
                           value_gold=next((i for i, v in enumerate(s['values']) if norm_text(v) == norm_text(s['reference_value'])), -1))
            reviewed_records.append(rec)
    write_jsonl(RUN_DIR / 'reviewed_examples.jsonl', reviewed_records)
    review_df = pd.DataFrame(review_rows)
    display(review_df)
    print(f'{len(review_df)} geprüfte Zustände, {int(review_df.Korrektur.sum())} Korrekturen '
          f'({review_df.Korrektur.mean():.0%} der Schritte).')

In [ ]:
# @title Adapter mit Korrekturen und Replay weitertrainieren
from peft import set_peft_model_state_dict
from safetensors.torch import load_file


class RecordItems(torch.utils.data.Dataset):
    # Wie Items, für Korrekturen + Replay.
    def __init__(self, records, value_records):
        self.records = records + [dict(r, _value=True) for r in value_records]
        random.Random(SEED).shuffle(self.records)
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        rng = random.Random(SEED * 100003 + i)
        item = value_item(TOKENIZER, r, rng) if r.get('_value') else action_item(TOKENIZER, r, rng)
        item['weight'] = 1.0
        return item


if RUN_CORRECTIONS:
    if 'v2' in model.peft_config:
        model.delete_adapter('v2')             # v2 wird ab hier nicht mehr gebraucht; weitertrainiert wird nur 'default'
        model.set_adapter('default')
    # Gleiches Tokenbudget wie im Training: sehr lange Seiten (z. B. die Länderliste mit ~37k Tokens) passen im
    # Training mit Gradienten nicht auf kleine GPUs und werden ausgelassen.
    n_tokens = [len(action_item(TOKENIZER, r)['ids']) for r in reviewed_records]
    corr_actions = [r for r, n in zip(reviewed_records, n_tokens) if n <= CFG['max_tokens']]
    skipped_long = len(reviewed_records) - len(corr_actions)
    print(f'{skipped_long} geprüfte Zustände über {CFG["max_tokens"]} Tokens ausgelassen (Tokenbudget des Profils).')
    corr_values = [r for r in corr_actions if r['kind'] == 'type' and r['value_gold'] >= 0]
    rng = random.Random(SEED + 1)
    replay = rng.sample(train_actions, min(len(train_actions), 3 * len(corr_actions)))
    replay_values = rng.sample(train_values, min(len(train_values), 3 * max(1, len(corr_values))))
    if (CONTINUED_DIR / 'adapter_model.safetensors').exists():
        set_peft_model_state_dict(model, load_file(CONTINUED_DIR / 'adapter_model.safetensors'))
        print('Weitertrainierter Adapter geladen:', CONTINUED_DIR)
    else:
        t0 = time.time()
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.enable_input_require_grads()
        model.train()
        cont = make_trainer(model, RecordItems(corr_actions + replay, corr_values + replay_values),
                            RUN_DIR / 'checkpoints-continued', lr=CFG['lr'] / 2, epochs=3, save_steps=10**9)
        wait_for_gpu(6)
        cont.train()
        cont.save_model(str(CONTINUED_DIR))
        TIMINGS['continue_s'] = time.time() - t0
        (RUN_DIR / 'continuation_recipe.json').write_text(json.dumps(dict(
            source_adapter=str(ADAPTER_DIR), reviewed_states=len(corr_actions), skipped_over_token_budget=skipped_long,
            corrections=int(sum(r['corrected'] for r in corr_actions)),
            value_examples=len(corr_values), replay=len(replay), replay_values=len(replay_values), epochs=3,
            lr=CFG['lr'] / 2, optimizer_steps=cont.state.global_step, seconds=TIMINGS['continue_s'], precision=PRECISION), indent=2))
        del cont
        gc.collect(); torch.cuda.empty_cache()
    model.eval()
    print(f'{len(corr_actions)} geprüfte Zustände + {len(replay)} Replay-Entscheidungen, '
          f'{len(corr_values)} + {len(replay_values)} Texteingaben.')

In [ ]:
# @title Zusammenfassung (nach der Korrektur: Prüfaufgaben, Korrekturaufgaben, Held-out-Regression)
if RUN_CORRECTIONS:
    live['eigenes+Korrektur'] = await suite('eval_continued', 'eigenes+Korrektur', EVAL_TASKS, True)
    corr_after = await suite('corr_continued', 'eigenes+Korrektur', CORRECTION_TASKS, True)
    reg = eval_records[:CFG['n_regression']]
    reg_after = cached_frame('heldout_continued', lambda: evaluate_actions(reg, True, 'eigenes+Korrektur'))
    reg_states = set(reg_after.state)

live_df = pd.concat([suite_frame(v, EVAL_TASKS) for v in live.values()])
rows = {
    'Held-out-Schrittgenauigkeit (n=%d)' % len(eval_records): {k: ev[k].correct.mean() for k in ev},
    'Held-out-Eingabetext (n=%d)' % len(eval_values): {k: evv[k].correct.mean() for k in evv},
    'Prüfaufgaben: Ziel erreicht': live_df.groupby('Bedingung').Erfolg.mean().to_dict(),
    'Prüfaufgaben: Daten korrekt': live_df.groupby('Bedingung').Daten_korrekt.mean().to_dict(),
}
if RUN_CORRECTIONS:
    corr_df = pd.concat([suite_frame(corr_before, CORRECTION_TASKS).assign(Bedingung='eigenes'),
                         suite_frame(corr_after, CORRECTION_TASKS).assign(Bedingung='eigenes+Korrektur')])
    rows['Held-out-Teilmenge (n=%d)' % len(reg)] = {**{k: ev[k][ev[k].state.isin(reg_states)].correct.mean() for k in ev},
                                                    'eigenes+Korrektur': reg_after.correct.mean()}
    rows['Korrekturaufgaben (gesehen): Ziel'] = corr_df.groupby('Bedingung').Erfolg.mean().to_dict()
columns = [c for c in ['Basis', 'v2 (veröffentlicht)', 'eigenes', 'eigenes+Korrektur']
           if any(c in r for r in rows.values())]
summary = pd.DataFrame(rows).T.reindex(columns=columns)
summary.to_csv(RUN_DIR / 'summary.csv')
display(summary.style.format('{:.1%}', na_rep='–'))
display(live_df.pivot_table(index='Aufgabe', columns='Bedingung', values='Erfolg', aggfunc='first'))
print(f'Referenz des vollen Laufs (BF16, A6000) auf den ersten {len(eval_records)} Held-out-Entscheidungen: '
      f'Basis {precision_check.loc["Basis", "Genauigkeit BF16-Referenz"]:.1%}, '
      f'v2 {precision_check.loc["v2 (veröffentlicht)", "Genauigkeit BF16-Referenz"]:.1%}; live Basis 6/16, v2 13/16.')

## 12 · Eigenen Adapter auf den Hub
Lädt den eigenen Adapter mit Tokenizer, `decision_contract.json` (Frage, Kandidaten-Labels, Basis-Commit, Held-out-Fingerabdruck),
`inference.py` (aus dem veröffentlichten Repo, fester Commit), `training/objective_filter.py`, Rezept, Logs, Auswertungen und einer
Modellkarte hoch, nach `continued/` die Variante nach dem Weitertraining. Braucht das Colab-Secret `HF_TOKEN` mit Schreibrecht;
Ziel ist ein **privates** Repo im eigenen Namensraum. Ohne Token bleibt alles lokal unter `RUN_DIR`.

In [ ]:
# @title Adapter hochladen
import tempfile
if not PUSH_ADAPTER:
    print('PUSH_ADAPTER ist aus. Adapter liegt in', ADAPTER_DIR)
elif not HF_TOKEN:
    print('Kein HF_TOKEN (Colab-Secret): kein Upload. Adapter liegt in', ADAPTER_DIR)
else:
    api = HfApi(token=HF_TOKEN)
    hub_repo = HUB_ADAPTER_REPO or f"{api.whoami()['name']}/decider-2b-a11y-crawler-lora-colab"
    if not of.hub_can_write(hub_repo, HF_TOKEN):
        print(f'Kein Schreibrecht für {hub_repo} (Token nur lesend?). Adapter liegt in', ADAPTER_DIR)
    else:
        stage = Path(tempfile.mkdtemp())
        shutil.copytree(ADAPTER_DIR, stage, dirs_exist_ok=True)
        contract = {**PUBLISHED_CONTRACT, 'version': f'colab-{PROFILE}-{PRECISION}', 'max_trained_tokens': CFG['max_tokens'],
                    'heldout_digest': HELDOUT_DIGEST}
        (stage / 'decision_contract.json').write_text(json.dumps(contract, indent=2))
        if (CONTINUED_DIR / 'adapter_model.safetensors').exists():
            shutil.copytree(CONTINUED_DIR, stage / 'continued')
            (stage / 'continued' / 'decision_contract.json').write_text(json.dumps(contract, indent=2))
            shutil.copyfile(RUN_DIR / 'continuation_recipe.json', stage / 'continued' / 'continuation_recipe.json')
        shutil.copyfile(hf_hub_download(MODEL_REPO, 'inference.py', revision=PUBLISHED_REVISION), stage / 'inference.py')
        (stage / 'training' / '02_data').mkdir(parents=True)
        shutil.copyfile(module_dir / 'objective_filter.py', stage / 'training' / '02_data' / 'objective_filter.py')
        for name in ['training_recipe.json', 'training_log.csv', 'val_log.jsonl', 'summary.csv', 'precision_check.csv',
                     'crawl_data_own.jsonl'] + [f'heldout_{k}.csv' for k in KEYS.values()]:
            if (RUN_DIR / name).exists():
                shutil.copyfile(RUN_DIR / name, stage / 'training' / name)

        def md_table(df):
            df = df.reset_index()
            rows = [' | '.join(map(str, df.columns)), ' | '.join(['---'] * len(df.columns))]
            rows += [' | '.join('–' if pd.isna(v) else f'{v:.1%}' if isinstance(v, float) else str(v) for v in r)
                     for r in df.itertuples(index=False)]
            return '\n'.join(f'| {r} |' for r in rows)

        (stage / 'README.md').write_text(f'''---
base_model: {MODEL_ID}
library_name: peft
license: apache-2.0
datasets: [{DATASET_ID}, {FILTER_REPO}]
tags: [lora, decider, web-agents, accessibility-tree, crawling]
---
# decider-2b a11y crawler (LoRA, Colab-Lauf, Profil `{PROFILE}`)

LoRA auf `{MODEL_ID}@{MODEL_SHA[:12]}`, trainiert mit der Colab-Fassung des Notebooks zu
[{MODEL_REPO}](https://huggingface.co/{MODEL_REPO}): {len(train_actions)} Aktions- und {len(train_values)} Texteingabe-Entscheidungen
aus NNetNav-live (Seiten bis {CFG['max_tokens']} Tokens, Auftragsfilter), {recipe['optimizer_steps']} Optimizer-Schritte,
{PRECISION.upper()} auf {GPU_NAME}, {recipe['seconds'] / 3600:.1f} h. Gleiche Eingabe, gleiche Kandidaten und gleiche `inference.py` wie v2.

## Ergebnisse (Held-out-Sites amazon.com, allrecipes.com, cambridge.org; Live-Aufgaben auf Scraping-Übungsseiten)

{md_table(summary)}
''')
        api.create_repo(hub_repo, private=HUB_ADAPTER_PRIVATE, exist_ok=True)
        info = api.upload_folder(repo_id=hub_repo, folder_path=str(stage),
                                 commit_message=f'Colab run {RUN_DIR.name}: {len(train_actions)} decisions')
        print('Hochgeladen:', info.commit_url if hasattr(info, 'commit_url') else info)

## 13 · Mitnehmen
Im Laufverzeichnis (auf Drive, wenn eingebunden) liegen: `adapter/` (bester Checkpoint nach Validierung) und `adapter-continued/`,
`training_recipe.json`, `training_log.csv`, `val_log.jsonl`, die gezogenen Entscheidungen (`data/`), alle Auswertungen als CSV/JSON,
die Crawl-Traces (`live_*.json`), die geprüften Korrekturen (`reviewed_examples.jsonl`) und die gesammelten Daten
(`crawl_data_own.jsonl`). Metadaten und Held-out-Auswahl von NNetNav liegen profilübergreifend unter `OUTPUT_ROOT/nnetnav/`.
Das 2B-Basismodell wird nicht kopiert; der Adapter nennt dessen Commit.

In [ ]:
# @title Laufzeiten und optional ZIP-Export
TIMINGS.setdefault('train_s', recipe['seconds'])
TIMINGS['peak_gib_process'] = torch.cuda.max_memory_allocated() / 2**30
(RUN_DIR / 'timings.json').write_text(json.dumps(TIMINGS, indent=2))
display(pd.Series(TIMINGS).round(1).to_frame('Wert'))

CREATE_ZIP = False  # @param {type:'boolean'}
if CREATE_ZIP:
    import zipfile
    zip_path = RUN_DIR.with_suffix('.zip')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in RUN_DIR.rglob('*'):
            if path.is_file() and not any(p.startswith('checkpoints') for p in path.relative_to(RUN_DIR).parts):
                archive.write(path, path.relative_to(RUN_DIR))
    print('Paket:', zip_path)

### Später wieder laden
```python
# Adapter (eigener aus RUN_DIR/adapter bzw. dem eigenen Hub-Repo, oder der veröffentlichte v2) mit inference.py
import importlib.util
from huggingface_hub import hf_hub_download
spec = importlib.util.spec_from_file_location('inference', hf_hub_download('johannhartmann/decider-2b-a11y-crawler-lora', 'inference.py'))
inference = importlib.util.module_from_spec(spec); spec.loader.exec_module(inference)
dec = inference.ActionDecider.from_pretrained('<dein-user>/decider-2b-a11y-crawler-lora-colab')
probs = dec.action_probs(state_text, [a['label'] for a in actions])   # state_text = observation_text(...)
```
Für Entscheidungen dieselben Bausteine wie hier verwenden: BrowserGym-Beobachtung, `candidates`, `observation_text`,
`score_options`. `inference.py` lädt in BF16; auf einer T4 `dtype=torch.float16` verwenden.

### Quellen
- [Decider-2B](https://huggingface.co/Mapika/decider-2b) · [Decider-Code](https://github.com/Mapika/decider)
- [NNetNav-Live](https://huggingface.co/datasets/stanfordnlp/nnetnav-live) · [NNetNav](https://github.com/MurtyShikhar/NNetNav)
  (AgentLab-Beobachtung mit `extract_visible_tag`, `extract_clickable_tag`; Historie aus `get_action_description_bgym`)
- [Auftragsfilter](https://huggingface.co/datasets/johannhartmann/nnetnav-live-objective-filter) (Claude Haiku 4.5 über [distilabel](https://github.com/argilla-io/distilabel))
- [Veröffentlichter Adapter v2](https://huggingface.co/johannhartmann/decider-2b-a11y-crawler-lora)
- [BrowserGym v0.13.3](https://github.com/ServiceNow/BrowserGym/tree/v0.13.3): `frame_mark_elements.js`, `observation.py`, `utils/obs.py`
- Übungsseiten: [Books to Scrape](https://books.toscrape.com), [Quotes to Scrape](https://quotes.toscrape.com),
  [Web Scraper Test Sites](https://webscraper.io/test-sites), [Scrape This Site](https://www.scrapethissite.com/pages/)
- [PEFT LoRA](https://huggingface.co/docs/peft/en/developer_guides/lora) · [flash-linear-attention](https://github.com/fla-org/flash-linear-attention)